In [ ]:
# ==============================================================
# R0 - Runtime / DuckDB lock-safe bootstrap
# ==============================================================
from pathlib import Path
import duckdb
import tomllib
import atexit

PROJECT_ROOT = Path.cwd()
CONFIG_TOML_PATH = PROJECT_ROOT / "benchmark_shared_config.toml"

if CONFIG_TOML_PATH.exists():
    with open(CONFIG_TOML_PATH, "rb") as f:
        SHARED_CONFIG = tomllib.load(f)
    print("Loaded shared config:", CONFIG_TOML_PATH)
else:
    SHARED_CONFIG = {
        "paths": {
            "data_dir": "content",
            "output_dir": "outputs_benchmark_survival",
            "tables_subdir": "tables",
            "metadata_subdir": "metadata",
            "data_output_subdir": "data",
            "duckdb_filename": "benchmark_survival.duckdb",
        },
        "benchmark": {
            "seed": 42,
            "test_size": 0.30,
            "early_window_weeks": 4,
            "main_enrollment_window_weeks": 4,
        },
    }
    print("Shared config TOML not found. Using defaults in-memory.")

paths_cfg = SHARED_CONFIG.get("paths", {})
SHARED_BENCHMARK_CONFIG = SHARED_CONFIG.get("benchmark", {})

def _resolve_project_path(raw_path: str) -> Path:
    p = Path(raw_path)
    return p if p.is_absolute() else PROJECT_ROOT / p

DATA_DIR = _resolve_project_path(paths_cfg.get("data_dir", "content"))
OUTPUT_DIR = _resolve_project_path(paths_cfg.get("output_dir", "outputs_benchmark_survival"))
TABLES_DIR = OUTPUT_DIR / paths_cfg.get("tables_subdir", "tables")
METADATA_DIR = OUTPUT_DIR / paths_cfg.get("metadata_subdir", "metadata")
DATA_OUTPUT_DIR = OUTPUT_DIR / paths_cfg.get("data_output_subdir", "data")
DUCKDB_PATH = OUTPUT_DIR / paths_cfg.get("duckdb_filename", "benchmark_survival.duckdb")

for p in [OUTPUT_DIR, TABLES_DIR, METADATA_DIR, DATA_OUTPUT_DIR]:
    p.mkdir(parents=True, exist_ok=True)

if "con" in globals():
    try:
        con.close()
        print("Closed previous DuckDB connection before reconnecting.")
    except Exception as _close_err:
        print(f"Warning while closing previous DuckDB connection: {_close_err}")

con = duckdb.connect(str(DUCKDB_PATH))

def _close_duckdb_connection() -> None:
    if "con" in globals():
        try:
            con.close()
            print("DuckDB connection closed.")
        except Exception:
            pass

if "_duckdb_close_registered" not in globals():
    atexit.register(_close_duckdb_connection)
    _duckdb_close_registered = True

print("Runtime context ready.")
print("- OUTPUT_DIR  :", OUTPUT_DIR)
print("- DUCKDB_PATH :", DUCKDB_PATH)

# G — Explainability and Paper Integration

This section consolidates the tuned-model explainability layer and the final export path used by the manuscript.

# G1 — Define Explainability Protocol

### Funcao: save_json

Definicao isolada de save_json para reutilizacao nas etapas seguintes.


In [ ]:
from pathlib import Path
import json

OUTPUT_DIR = Path("outputs_benchmark_survival")
TABLES_DIR = OUTPUT_DIR / "tables"
METADATA_DIR = OUTPUT_DIR / "metadata"
DATA_DIR = OUTPUT_DIR / "data"
TABLES_DIR.mkdir(parents=True, exist_ok=True)
METADATA_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_SEED = 42
HORIZONS_WEEKS = [10, 20, 30]
HORIZON_WEEKS = list(HORIZONS_WEEKS)
CALIBRATION_BINS = 10


def save_json(obj: dict, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as file_handle:
        json.dump(obj, file_handle, indent=2)


try:
    import pyarrow

    _original_unregister_extension_type = pyarrow.unregister_extension_type

    def _safe_unregister_extension_type(name):
        try:
            return _original_unregister_extension_type(name)
        except Exception:
            if name == "arrow.py_extension_type":
                return None
            raise

    pyarrow.unregister_extension_type = _safe_unregister_extension_type
except Exception:
    pass


try:
    import pandas as pd

    neural_ready_paths = [
        DATA_DIR / "pp_neural_hazard_ready_train.parquet",
        DATA_DIR / "pp_neural_hazard_ready_test.parquet",
        DATA_DIR / "pp_neural_hazard_ready_train.csv",
        DATA_DIR / "pp_neural_hazard_ready_test.csv",
    ]
    patched_paths = []
    for dataset_path in neural_ready_paths:
        if not dataset_path.exists():
            continue
        if dataset_path.suffix == ".parquet":
            dataset_df = pd.read_parquet(dataset_path)
        else:
            dataset_df = pd.read_csv(dataset_path)
        if "time_for_split" not in dataset_df.columns and "t_final_week" in dataset_df.columns:
            dataset_df["time_for_split"] = dataset_df["t_final_week"]
            if dataset_path.suffix == ".parquet":
                dataset_df.to_parquet(dataset_path, index=False)
            else:
                dataset_df.to_csv(dataset_path, index=False)
            patched_paths.append(str(dataset_path))
    if patched_paths:
        print("Patched neural ready datasets with time_for_split:")
        for patched_path in patched_paths:
            print("-", patched_path)
except Exception as exc:
    print(f"Neural dataset schema patch skipped: {exc}")


try:
    import sklearn
    from sklearn.compose import _column_transformer
    if not hasattr(_column_transformer, "_RemainderColsList"):
        class _RemainderColsList(list):
            pass
        _column_transformer._RemainderColsList = _RemainderColsList
    print("Explainability bootstrap ready.", sklearn.__version__)
except Exception as exc:
    print(f"Explainability bootstrap ready without sklearn patch: {exc}")

In [ ]:
# ==============================================================
# G1 — Define Explainability Protocol
# --------------------------------------------------------------
# Purpose:
#   Define the explainability study design for the tuned models
#   already established in the benchmark.
#
# Methodological note:
#   This step does not train any model and does not compute any
#   explanation yet. It only formalizes:
#     - which models are included,
#     - which explainability methods will be used,
#     - which outputs are expected,
#     - and how the results should be interpreted.
#
# Scope:
#   Explainability will be performed for the tuned versions only:
#     - linear_tuned
#     - neural_tuned
#     - cox_tuned
#     - deepsurv_tuned
# ==============================================================

print("\n" + "=" * 70)
print("G1 — Define Explainability Protocol")
print("=" * 70)
print("Methodological note: this step defines the explainability study only.")
print("No model is trained and no explanation is computed here.")

# ------------------------------
# 1) Basic checks
# ------------------------------
required_names = ["TABLES_DIR", "METADATA_DIR", "save_json"]
missing_names = [name for name in required_names if name not in globals()]
if missing_names:
    raise NameError(
        "Missing required objects from previous cells: "
        + ", ".join(missing_names)
        + ". Please run prior setup cells first."
    )

import pandas as pd

# ------------------------------
# 2) Model registry
# ------------------------------
EXPLAINABILITY_MODEL_REGISTRY = [
    {
        "model_id": "linear_tuned",
        "display_name": "Linear Discrete-Time Hazard (Tuned)",
        "family": "discrete_time_linear",
        "data_level": "person_period",
        "primary_explainability_method": "coefficient_analysis",
        "secondary_explainability_method": "odds_ratio_interpretation",
        "local_explanation_planned": False,
        "global_explanation_planned": True,
        "positioning_note": "Transparent linear discrete-time benchmark."
    },
    {
        "model_id": "neural_tuned",
        "display_name": "Neural Discrete-Time Survival (Tuned)",
        "family": "discrete_time_neural",
        "data_level": "person_period",
        "primary_explainability_method": "permutation_importance",
        "secondary_explainability_method": "feature_block_importance_summary",
        "local_explanation_planned": False,
        "global_explanation_planned": True,
        "positioning_note": "Flexible nonlinear discrete-time benchmark."
    },
    {
        "model_id": "cox_tuned",
        "display_name": "Cox Comparable (Tuned)",
        "family": "continuous_time_cox",
        "data_level": "enrollment",
        "primary_explainability_method": "hazard_ratio_analysis",
        "secondary_explainability_method": "coefficient_ranking",
        "local_explanation_planned": False,
        "global_explanation_planned": True,
        "positioning_note": "Interpretable continuous-time survival benchmark."
    },
    {
        "model_id": "deepsurv_tuned",
        "display_name": "DeepSurv (Tuned)",
        "family": "continuous_time_deepsurv",
        "data_level": "enrollment",
        "primary_explainability_method": "permutation_importance",
        "secondary_explainability_method": "feature_block_importance_summary",
        "local_explanation_planned": False,
        "global_explanation_planned": True,
        "positioning_note": "Nonlinear continuous-time survival benchmark."
    },
]

explainability_model_registry_df = pd.DataFrame(EXPLAINABILITY_MODEL_REGISTRY)

# ------------------------------
# 3) Feature-block registry
# ------------------------------
EXPLAINABILITY_FEATURE_BLOCKS = [
    {
        "block_id": "static_structural",
        "block_label": "Static structural covariates",
        "applies_to": "all_families",
        "examples": "gender, region, highest_education, imd_band, age_band, num_of_prev_attempts, studied_credits, disability",
        "interpretation_goal": "Assess contribution of background and structural covariates."
    },
    {
        "block_id": "dynamic_temporal_behavioral",
        "block_label": "Dynamic temporal behavioral signals",
        "applies_to": "discrete_time_only",
        "examples": "total_clicks_week, active_this_week, n_vle_rows_week, n_distinct_sites_week, cum_clicks_until_t, recency, streak",
        "interpretation_goal": "Assess contribution of week-by-week engagement signals."
    },
    {
        "block_id": "discrete_time_index",
        "block_label": "Discrete-time structural index",
        "applies_to": "discrete_time_only",
        "examples": "week",
        "interpretation_goal": "Represent time index, not behavioral signal."
    },
    {
        "block_id": "early_window_behavior",
        "block_label": "Early-window behavioral summaries",
        "applies_to": "continuous_time_only",
        "examples": "clicks_first_4_weeks, active_weeks_first_4, mean_clicks_first_4_weeks",
        "interpretation_goal": "Assess contribution of compressed early-course behavior."
    },
]

explainability_feature_blocks_df = pd.DataFrame(EXPLAINABILITY_FEATURE_BLOCKS)

# ------------------------------
# 4) Expected outputs
# ------------------------------
EXPLAINABILITY_OUTPUTS = [
    {
        "output_id": "global_feature_ranking",
        "description": "Rank features by global importance within each tuned family.",
        "planned_for": "all_models"
    },
    {
        "output_id": "feature_block_summary",
        "description": "Summarize importance patterns at the feature-block level.",
        "planned_for": "all_models"
    },
    {
        "output_id": "signed_effect_table",
        "description": "Report signed coefficient or hazard-ratio direction when directly available.",
        "planned_for": "linear_tuned_and_cox_tuned"
    },
    {
        "output_id": "cross_family_explainability_summary",
        "description": "Compare whether the strongest drivers are consistent across families.",
        "planned_for": "final_consolidation"
    },
]

explainability_outputs_df = pd.DataFrame(EXPLAINABILITY_OUTPUTS)

# ------------------------------
# 5) Protocol
# ------------------------------
EXPLAINABILITY_PROTOCOL = {
    "scope": "tuned_models_only",
    "included_models": [row["model_id"] for row in EXPLAINABILITY_MODEL_REGISTRY],
    "main_goals": [
        "Identify which individual features are most influential within each tuned model family.",
        "Compare the dominant explanatory signals across linear, neural, Cox, and DeepSurv families.",
        "Connect explainability findings back to the ablation results."
    ],
    "global_vs_local_policy": {
        "global_explanations": True,
        "local_explanations": False,
        "rationale": (
            "The main goal of this benchmark stage is model-level interpretation, "
            "not case-level explanation."
        )
    },
    "method_policy_by_family": {
        "discrete_time_linear": {
            "primary": "coefficient_analysis",
            "secondary": "odds_ratio_interpretation"
        },
        "discrete_time_neural": {
            "primary": "permutation_importance",
            "secondary": "feature_block_importance_summary"
        },
        "continuous_time_cox": {
            "primary": "hazard_ratio_analysis",
            "secondary": "coefficient_ranking"
        },
        "continuous_time_deepsurv": {
            "primary": "permutation_importance",
            "secondary": "feature_block_importance_summary"
        }
    },
    "interpretation_rules": {
        "large_positive_signed_effect": "Associated with increased risk when sign-based methods apply.",
        "large_negative_signed_effect": "Associated with reduced risk when sign-based methods apply.",
        "high_permutation_importance": "Model performance depends strongly on that feature.",
        "consistency_with_ablation": (
            "If an important feature belongs to a behavior-derived block, the result should align "
            "with the ablation conclusion that behavioral information dominates."
        )
    },
    "limitations_to_document": [
        "Permutation importance is global and model-dependent, not causal.",
        "Coefficient and hazard-ratio interpretations apply only to model families with directly interpretable parameters.",
        "Explainability describes how the model uses information, not whether the relationships are causal."
    ],
    "paper_positioning_note": (
        "Explainability is treated here as a post-benchmark interpretive layer, intended to explain "
        "why the tuned models behave as they do after ranking has already been established."
    )
}

# ------------------------------
# 6) Save outputs
# ------------------------------
model_registry_path = TABLES_DIR / "table_explainability_model_registry.csv"
feature_blocks_path = TABLES_DIR / "table_explainability_feature_blocks.csv"
outputs_path = TABLES_DIR / "table_explainability_outputs.csv"
config_path = METADATA_DIR / "explainability_config.json"

explainability_model_registry_df.to_csv(model_registry_path, index=False)
explainability_feature_blocks_df.to_csv(feature_blocks_path, index=False)
explainability_outputs_df.to_csv(outputs_path, index=False)
save_json(EXPLAINABILITY_PROTOCOL, config_path)

# ------------------------------
# 7) Output for feedback
# ------------------------------
print("\nExplainability model registry:")
display(explainability_model_registry_df)

print("\nExplainability feature blocks:")
display(explainability_feature_blocks_df)

print("\nExplainability planned outputs:")
display(explainability_outputs_df)

print("\nExplainability protocol summary:")
display(pd.DataFrame([{
    "scope": EXPLAINABILITY_PROTOCOL["scope"],
    "included_models": ", ".join(EXPLAINABILITY_PROTOCOL["included_models"]),
    "global_explanations": EXPLAINABILITY_PROTOCOL["global_vs_local_policy"]["global_explanations"],
    "local_explanations": EXPLAINABILITY_PROTOCOL["global_vs_local_policy"]["local_explanations"],
    "paper_positioning_note": EXPLAINABILITY_PROTOCOL["paper_positioning_note"],
}]))

print("\nSaved:")
print("-", model_registry_path.resolve())
print("-", feature_blocks_path.resolve())
print("-", outputs_path.resolve())
print("-", config_path.resolve())

# G2 — Explainability for Linear Tuned

### Funcao: infer_block

Definicao isolada de infer_block para reutilizacao nas etapas seguintes.


In [ ]:
# ==============================================================
# G2 — Explainability for Linear Tuned
# --------------------------------------------------------------
# Purpose:
#   Produce global explainability outputs for the tuned linear
#   discrete-time hazard model.
#
# Methodological note:
#   This step uses direct parameter interpretation because the
#   tuned linear model is intrinsically interpretable.
#
# Outputs:
#   - feature-level coefficient table
#   - odds-ratio table
#   - block-level summary
# ==============================================================

print("\n" + "=" * 70)
print("G2 — Explainability for Linear Tuned")
print("=" * 70)
print("Methodological note: this step computes global explainability")
print("for the tuned linear discrete-time hazard model.")

# ------------------------------
# 1) Basic checks
# ------------------------------
required_names = ["OUTPUT_DIR", "TABLES_DIR", "METADATA_DIR"]
missing_names = [name for name in required_names if name not in globals()]
if missing_names:
    raise NameError(
        "Missing required objects from previous cells: "
        + ", ".join(missing_names)
        + ". Please run prior setup cells first."
    )

from pathlib import Path
import numpy as np
import pandas as pd
import joblib

# ------------------------------
# 2) Paths
# ------------------------------
MODEL_DIR = OUTPUT_DIR / "models"

model_path = MODEL_DIR / "linear_discrete_time_hazard_tuned.joblib"
if not model_path.exists():
    raise FileNotFoundError(f"Tuned model file not found: {model_path}")

preprocessor_path = MODEL_DIR / "linear_discrete_time_preprocessor.joblib"

if not model_path.exists():
    raise FileNotFoundError(f"Model file not found: {model_path}")
if not preprocessor_path.exists():
    raise FileNotFoundError(f"Preprocessor file not found: {preprocessor_path}")

# ------------------------------
# 3) Load artifacts
# ------------------------------
linear_model = joblib.load(model_path)
linear_preprocessor = joblib.load(preprocessor_path)

# ------------------------------
# 4) Recover transformed feature names
# ------------------------------
if not hasattr(linear_preprocessor, "get_feature_names_out"):
    raise AttributeError("The loaded preprocessor does not expose get_feature_names_out().")

feature_names_out = list(linear_preprocessor.get_feature_names_out())

if not hasattr(linear_model, "coef_"):
    raise AttributeError("The loaded linear model does not expose coef_.")

coefs = linear_model.coef_.reshape(-1)

if len(feature_names_out) != len(coefs):
    raise ValueError(
        f"Mismatch between transformed feature names ({len(feature_names_out)}) "
        f"and coefficients ({len(coefs)})."
    )

# ------------------------------
# 5) Feature-level explainability table
# ------------------------------
explain_df = pd.DataFrame({
    "feature_name_out": feature_names_out,
    "coefficient": coefs,
    "abs_coefficient": np.abs(coefs),
    "odds_ratio": np.exp(coefs),
})

def infer_block(feature_name: str) -> str:
    if feature_name.startswith("num__week"):
        return "discrete_time_index"
    if feature_name.startswith("num__total_clicks_week"):
        return "dynamic_temporal_behavioral"
    if feature_name.startswith("num__active_this_week"):
        return "dynamic_temporal_behavioral"
    if feature_name.startswith("num__n_vle_rows_week"):
        return "dynamic_temporal_behavioral"
    if feature_name.startswith("num__n_distinct_sites_week"):
        return "dynamic_temporal_behavioral"
    if feature_name.startswith("num__cum_clicks_until_t"):
        return "dynamic_temporal_behavioral"
    if feature_name.startswith("num__recency"):
        return "dynamic_temporal_behavioral"
    if feature_name.startswith("num__streak"):
        return "dynamic_temporal_behavioral"
    if feature_name.startswith("num__num_of_prev_attempts"):
        return "static_structural"
    if feature_name.startswith("num__studied_credits"):
        return "static_structural"
    if feature_name.startswith("cat__gender_"):
        return "static_structural"
    if feature_name.startswith("cat__region_"):
        return "static_structural"
    if feature_name.startswith("cat__highest_education_"):
        return "static_structural"
    if feature_name.startswith("cat__imd_band_"):
        return "static_structural"
    if feature_name.startswith("cat__age_band_"):
        return "static_structural"
    if feature_name.startswith("cat__disability_"):
        return "static_structural"
    return "other"

explain_df["feature_block"] = explain_df["feature_name_out"].apply(infer_block)

explain_df["effect_direction"] = np.where(
    explain_df["coefficient"] > 0,
    "increases_log_odds_of_weekly_event",
    np.where(
        explain_df["coefficient"] < 0,
        "decreases_log_odds_of_weekly_event",
        "neutral"
    )
)

explain_df_sorted_abs = explain_df.sort_values(
    by="abs_coefficient", ascending=False
).reset_index(drop=True)

explain_df_sorted_signed = explain_df.sort_values(
    by="coefficient", ascending=False
).reset_index(drop=True)

# ------------------------------
# 6) Block-level summary
# ------------------------------
block_summary_df = (
    explain_df.groupby("feature_block", as_index=False)
    .agg(
        n_features=("feature_name_out", "count"),
        mean_abs_coefficient=("abs_coefficient", "mean"),
        median_abs_coefficient=("abs_coefficient", "median"),
        max_abs_coefficient=("abs_coefficient", "max"),
        mean_coefficient=("coefficient", "mean"),
    )
    .sort_values(by="mean_abs_coefficient", ascending=False)
    .reset_index(drop=True)
)

# ------------------------------
# 7) Top positive / negative effects
# ------------------------------
top_positive_df = explain_df.sort_values(
    by="coefficient", ascending=False
).head(15).reset_index(drop=True)

top_negative_df = explain_df.sort_values(
    by="coefficient", ascending=True
).head(15).reset_index(drop=True)

# ------------------------------
# 8) Save outputs
# ------------------------------
feature_table_path = TABLES_DIR / "table_linear_explainability_feature_coefficients.csv"
signed_table_path = TABLES_DIR / "table_linear_explainability_signed_effects.csv"
block_summary_path = TABLES_DIR / "table_linear_explainability_block_summary.csv"
top_positive_path = TABLES_DIR / "table_linear_explainability_top_positive.csv"
top_negative_path = TABLES_DIR / "table_linear_explainability_top_negative.csv"
config_path = METADATA_DIR / "linear_explainability_summary.json"

explain_df_sorted_abs.to_csv(feature_table_path, index=False)
explain_df_sorted_signed.to_csv(signed_table_path, index=False)
block_summary_df.to_csv(block_summary_path, index=False)
top_positive_df.to_csv(top_positive_path, index=False)
top_negative_df.to_csv(top_negative_path, index=False)

save_json(
    {
        "model_id": "linear_tuned",
        "model_file_used": str(model_path),
        "preprocessor_file_used": str(preprocessor_path),
        "n_transformed_features": int(len(feature_names_out)),
        "top_feature_by_abs_coef": explain_df_sorted_abs.iloc[0]["feature_name_out"],
        "top_feature_block_by_mean_abs_coef": block_summary_df.iloc[0]["feature_block"],
    },
    config_path,
)

# ------------------------------
# 9) Output for feedback
# ------------------------------
print("\nTop features by absolute coefficient:")
display(explain_df_sorted_abs.head(20))

print("\nTop positive effects:")
display(top_positive_df)

print("\nTop negative effects:")
display(top_negative_df)

print("\nFeature-block summary:")
display(block_summary_df)

print("\nSaved:")
print("-", feature_table_path.resolve())
print("-", signed_table_path.resolve())
print("-", block_summary_path.resolve())
print("-", top_positive_path.resolve())
print("-", top_negative_path.resolve())
print("-", config_path.resolve())

# G3 — Explainability for Neural Tuned (Revised v2)

In [ ]:
# ==============================================================
# G3 — Explainability for Neural Tuned (Revised v2)
# --------------------------------------------------------------
# Purpose:
#   Produce global explainability outputs for the tuned neural
#   discrete-time survival model using GROUPED permutation
#   importance at the original-feature level.
#
# Methodological note:
#   This revised version corrects the prior approach by avoiding
#   permutation at the one-hot encoded column level.
#
#   Instead, permutation is applied by original feature/group:
#   - each numeric feature is permuted as one group
#   - each categorical feature is permuted jointly across all
#     derived one-hot columns through raw-data permutation followed
#     by preprocessing
#
# This yields feature-level explainability that is better aligned
# with the original data semantics and with the ablation study.
# ==============================================================

print("\n" + "=" * 70)
print("G3 — Explainability for Neural Tuned (Revised v2)")
print("=" * 70)
print("Methodological note: this step computes global explainability")
print("for the tuned neural discrete-time survival model using GROUPED")
print("permutation importance at the original-feature level.")

# ------------------------------
# 1) Basic checks
# ------------------------------
required_names = [
    "OUTPUT_DIR", "TABLES_DIR", "METADATA_DIR",
    "RANDOM_SEED", "HORIZONS_WEEKS", "save_json"
]
missing_names = [name for name in required_names if name not in globals()]
if missing_names:
    raise NameError(
        "Missing required objects from previous cells: "
        + ", ".join(missing_names)
        + ". Please run prior setup cells first."
    )

from pathlib import Path
import numpy as np
import pandas as pd
import scipy
import torch
import torchtuples as tt
import joblib

from sklearn.metrics import roc_auc_score

try:
    from pycox.evaluation import EvalSurv
    PYCOX_AVAILABLE = True
except Exception:
    PYCOX_AVAILABLE = False

if not PYCOX_AVAILABLE:
    raise ImportError("pycox is required for P34.")

# ------------------------------
# 2) Compatibility patch for SciPy / PyCox
# ------------------------------
try:
    if not hasattr(scipy.integrate, "simps") and hasattr(scipy.integrate, "simpson"):
        def _simps_compat(y, x=None, dx=1.0, axis=-1, even=None):
            return scipy.integrate.simpson(y, x=x, dx=dx, axis=axis)
        scipy.integrate.simps = _simps_compat
except Exception:
    pass

# ------------------------------
# 3) Paths
# ------------------------------
MODEL_DIR = OUTPUT_DIR / "models"
DATA_DIR = OUTPUT_DIR / "data"

preprocessor_path = MODEL_DIR / "neural_discrete_time_preprocessor.joblib"
train_data_path = DATA_DIR / "pp_neural_hazard_ready_train.parquet"
test_data_path = DATA_DIR / "pp_neural_hazard_ready_test.parquet"

if not train_data_path.exists():
    train_data_path = DATA_DIR / "pp_neural_hazard_ready_train.csv"
if not test_data_path.exists():
    test_data_path = DATA_DIR / "pp_neural_hazard_ready_test.csv"

if not preprocessor_path.exists():
    raise FileNotFoundError(f"Preprocessor file not found: {preprocessor_path}")
if not train_data_path.exists():
    raise FileNotFoundError(f"Train data file not found: {train_data_path}")
if not test_data_path.exists():
    raise FileNotFoundError(f"Test data file not found: {test_data_path}")

# ------------------------------
# 4) Load artifacts
# ------------------------------
neural_preprocessor = joblib.load(preprocessor_path)

if str(train_data_path).endswith(".parquet"):
    neural_train_df = pd.read_parquet(train_data_path)
else:
    neural_train_df = pd.read_csv(train_data_path)

if str(test_data_path).endswith(".parquet"):
    neural_test_df = pd.read_parquet(test_data_path)
else:
    neural_test_df = pd.read_csv(test_data_path)

# ------------------------------
# 5) Column definitions
# ------------------------------
AUX_DISCRETE = [
    "enrollment_id",
    "id_student",
    "code_module",
    "code_presentation",
    "event_observed",
    "t_event_week",
    "t_final_week",
    "used_zero_week_fallback_for_censoring",
    "split",
    "time_for_split",
    "time_bucket",
    "event_time_bucket_label",
]

TARGET_COL = "event_t"

FEATURE_GROUPS = [
    "gender",
    "region",
    "highest_education",
    "imd_band",
    "age_band",
    "disability",
    "num_of_prev_attempts",
    "studied_credits",
    "week",
    "total_clicks_week",
    "active_this_week",
    "n_vle_rows_week",
    "n_distinct_sites_week",
    "cum_clicks_until_t",
    "recency",
    "streak",
]


### Funcao: get_feature_columns

Definicao isolada de get_feature_columns para reutilizacao nas etapas seguintes.


In [ ]:

def get_feature_columns(df: pd.DataFrame):
    excluded = set(AUX_DISCRETE + [TARGET_COL])
    return [c for c in df.columns if c not in excluded]


In [ ]:

feature_cols_raw = get_feature_columns(neural_train_df)

missing_expected_features = [c for c in FEATURE_GROUPS if c not in feature_cols_raw]
if missing_expected_features:
    raise ValueError(
        f"Missing expected feature groups in test/train data: {missing_expected_features}"
    )

# transformed design
X_train = neural_preprocessor.transform(neural_train_df[feature_cols_raw])
X_test = neural_preprocessor.transform(neural_test_df[feature_cols_raw])

if hasattr(X_train, "toarray"):
    X_train_dense = X_train.toarray().astype(np.float32)
    X_test_dense = X_test.toarray().astype(np.float32)
else:
    X_train_dense = np.asarray(X_train).astype(np.float32)
    X_test_dense = np.asarray(X_test).astype(np.float32)

y_train = neural_train_df[TARGET_COL].to_numpy().astype(np.float32)

# ------------------------------
# 6) Refit tuned neural model
# ------------------------------
torch.manual_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

net = tt.practical.MLPVanilla(
    in_features=X_train_dense.shape[1],
    num_nodes=[128, 64],
    out_features=1,
    batch_norm=True,
    dropout=0.1,
    output_bias=False,
)

model = tt.Model(net, torch.nn.BCEWithLogitsLoss(), tt.optim.AdamW)
model.optimizer.set_lr(5e-4)

_ = model.fit(
    X_train_dense,
    y_train.reshape(-1, 1),
    batch_size=256,
    epochs=25,
    verbose=False,
)


### Funcao: get_surv_at_horizon

Definicao isolada de get_surv_at_horizon para reutilizacao nas etapas seguintes.


In [ ]:

# ------------------------------
# 7) Evaluation helper
# ------------------------------
def get_surv_at_horizon(surv_frame: pd.DataFrame, h: int) -> pd.Series:
    idx = np.asarray(surv_frame.index, dtype=float)
    pos = np.searchsorted(idx, float(h), side="right") - 1
    if pos < 0:
        return pd.Series(np.ones(surv_frame.shape[1]), index=surv_frame.columns)
    return surv_frame.iloc[pos]


### Funcao: evaluate_discrete_survival_from_hazard

Definicao isolada de evaluate_discrete_survival_from_hazard para reutilizacao nas etapas seguintes.


In [ ]:

def evaluate_discrete_survival_from_hazard(test_pred_df: pd.DataFrame, horizons: list[int]):
    test_pred_df = test_pred_df.sort_values(["enrollment_id", "week"]).copy()
    test_pred_df["pred_survival"] = test_pred_df.groupby("enrollment_id")["pred_hazard"].transform(
        lambda s: (1.0 - s).cumprod()
    )
    test_pred_df["pred_risk"] = 1.0 - test_pred_df["pred_survival"]

    truth_test = (
        test_pred_df.groupby("enrollment_id", as_index=False)
        .agg(
            event=("event_observed", "max"),
            duration=("time_for_split", "max"),
        )
    )

    surv_wide = (
        test_pred_df[["enrollment_id", "week", "pred_survival"]]
        .drop_duplicates(subset=["enrollment_id", "week"])
        .pivot(index="week", columns="enrollment_id", values="pred_survival")
        .sort_index()
    )

    max_week_test = int(test_pred_df["week"].max())
    full_week_index = pd.Index(np.arange(0, max_week_test + 1), name="week")
    surv_wide = surv_wide.reindex(full_week_index).ffill().fillna(1.0)

    surv_df = surv_wide.copy()
    surv_df.columns.name = "enrollment_id"

    durations_test = truth_test["duration"].astype(int).to_numpy()
    events_test = truth_test["event"].astype(int).to_numpy()

    eval_surv = EvalSurv(
        surv=surv_df,
        durations=durations_test,
        events=events_test,
        censor_surv="km",
    )

    try:
        max_requested_horizon = int(max(horizons))
        ibs_grid = np.arange(1, max_requested_horizon + 1, dtype=int)
        ibs_value = float(eval_surv.integrated_brier_score(ibs_grid))
    except Exception:
        ibs_value = np.nan

    risk_auc_rows = []
    for h in horizons:
        pred_surv_h = get_surv_at_horizon(surv_df, h)
        pred_risk_h = 1.0 - pred_surv_h

        eval_df = truth_test.copy()
        eval_df["pred_risk_h"] = eval_df["enrollment_id"].map(pred_risk_h.to_dict())

        eval_df["is_evaluable_at_h"] = (
            ((eval_df["event"] == 1) & (eval_df["duration"] <= h)) |
            (eval_df["duration"] >= h)
        ).astype(int)

        eval_df = eval_df[eval_df["is_evaluable_at_h"] == 1].copy()
        eval_df["observed_event_by_h"] = ((eval_df["event"] == 1) & (eval_df["duration"] <= h)).astype(int)

        if eval_df["observed_event_by_h"].nunique() >= 2:
            risk_auc = roc_auc_score(eval_df["observed_event_by_h"], eval_df["pred_risk_h"])
        else:
            risk_auc = np.nan

        risk_auc_rows.append({
            "horizon_week": h,
            "risk_auc_at_horizon": float(risk_auc) if pd.notna(risk_auc) else np.nan,
        })

    return {
        "ibs": ibs_value,
        "risk_auc_df": pd.DataFrame(risk_auc_rows),
    }


### Funcao: predict_hazard_from_raw_df

Definicao isolada de predict_hazard_from_raw_df para reutilizacao nas etapas seguintes.


In [ ]:

def predict_hazard_from_raw_df(raw_df: pd.DataFrame) -> np.ndarray:
    X = neural_preprocessor.transform(raw_df[feature_cols_raw])
    if hasattr(X, "toarray"):
        X_dense = X.toarray().astype(np.float32)
    else:
        X_dense = np.asarray(X).astype(np.float32)

    logits = model.predict(X_dense).reshape(-1)
    pred_hazard = 1.0 / (1.0 + np.exp(-logits))
    return np.clip(pred_hazard, 1e-8, 1 - 1e-8)


### Funcao: infer_block

Definicao isolada de infer_block para reutilizacao nas etapas seguintes.


In [ ]:

def infer_block(feature_name: str) -> str:
    if feature_name == "week":
        return "discrete_time_index"
    if feature_name in [
        "total_clicks_week",
        "active_this_week",
        "n_vle_rows_week",
        "n_distinct_sites_week",
        "cum_clicks_until_t",
        "recency",
        "streak",
    ]:
        return "dynamic_temporal_behavioral"
    return "static_structural"


In [ ]:

# ------------------------------
# 8) Baseline performance
# ------------------------------
baseline_test_pred_df = neural_test_df.copy()
baseline_test_pred_df["pred_hazard"] = predict_hazard_from_raw_df(neural_test_df)

baseline_eval = evaluate_discrete_survival_from_hazard(
    baseline_test_pred_df,
    HORIZONS_WEEKS
)

baseline_ibs = baseline_eval["ibs"]
baseline_risk_auc_df = baseline_eval["risk_auc_df"].copy()

baseline_risk_auc_h10 = float(
    baseline_risk_auc_df.loc[baseline_risk_auc_df["horizon_week"] == 10, "risk_auc_at_horizon"].iloc[0]
)
baseline_risk_auc_h20 = float(
    baseline_risk_auc_df.loc[baseline_risk_auc_df["horizon_week"] == 20, "risk_auc_at_horizon"].iloc[0]
)
baseline_risk_auc_h30 = float(
    baseline_risk_auc_df.loc[baseline_risk_auc_df["horizon_week"] == 30, "risk_auc_at_horizon"].iloc[0]
)

# ------------------------------
# 9) GROUPED permutation importance
# ------------------------------
rng = np.random.default_rng(RANDOM_SEED)
importance_rows = []

for feat in FEATURE_GROUPS:
    perm_df = neural_test_df.copy()

    shuffled = perm_df[feat].to_numpy(copy=True)
    rng.shuffle(shuffled)
    perm_df[feat] = shuffled

    perm_test_pred_df = perm_df.copy()
    perm_test_pred_df["pred_hazard"] = predict_hazard_from_raw_df(perm_df)

    perm_eval = evaluate_discrete_survival_from_hazard(
        perm_test_pred_df,
        HORIZONS_WEEKS
    )

    perm_ibs = perm_eval["ibs"]
    perm_risk_auc_df = perm_eval["risk_auc_df"].copy()

    perm_risk_auc_h10 = float(
        perm_risk_auc_df.loc[perm_risk_auc_df["horizon_week"] == 10, "risk_auc_at_horizon"].iloc[0]
    )
    perm_risk_auc_h20 = float(
        perm_risk_auc_df.loc[perm_risk_auc_df["horizon_week"] == 20, "risk_auc_at_horizon"].iloc[0]
    )
    perm_risk_auc_h30 = float(
        perm_risk_auc_df.loc[perm_risk_auc_df["horizon_week"] == 30, "risk_auc_at_horizon"].iloc[0]
    )

    importance_rows.append({
        "feature_name_original": feat,
        "baseline_ibs": baseline_ibs,
        "permuted_ibs": perm_ibs,
        "delta_ibs": perm_ibs - baseline_ibs if pd.notna(perm_ibs) and pd.notna(baseline_ibs) else np.nan,
        "baseline_risk_auc_h10": baseline_risk_auc_h10,
        "permuted_risk_auc_h10": perm_risk_auc_h10,
        "delta_risk_auc_h10": perm_risk_auc_h10 - baseline_risk_auc_h10 if pd.notna(perm_risk_auc_h10) else np.nan,
        "baseline_risk_auc_h20": baseline_risk_auc_h20,
        "permuted_risk_auc_h20": perm_risk_auc_h20,
        "delta_risk_auc_h20": perm_risk_auc_h20 - baseline_risk_auc_h20 if pd.notna(perm_risk_auc_h20) else np.nan,
        "baseline_risk_auc_h30": baseline_risk_auc_h30,
        "permuted_risk_auc_h30": perm_risk_auc_h30,
        "delta_risk_auc_h30": perm_risk_auc_h30 - baseline_risk_auc_h30 if pd.notna(perm_risk_auc_h30) else np.nan,
    })

importance_df = pd.DataFrame(importance_rows)
importance_df["importance_score_ibs"] = importance_df["delta_ibs"]
importance_df["importance_score_risk_auc_h10"] = -importance_df["delta_risk_auc_h10"]
importance_df["importance_score_risk_auc_h20"] = -importance_df["delta_risk_auc_h20"]
importance_df["importance_score_risk_auc_h30"] = -importance_df["delta_risk_auc_h30"]
importance_df["mean_importance_score_auc"] = importance_df[
    ["importance_score_risk_auc_h10", "importance_score_risk_auc_h20", "importance_score_risk_auc_h30"]
].mean(axis=1)
importance_df["feature_block"] = importance_df["feature_name_original"].apply(infer_block)

importance_sorted_df = importance_df.sort_values(
    by=["importance_score_ibs", "mean_importance_score_auc"],
    ascending=[False, False]
).reset_index(drop=True)

# ------------------------------
# 10) Block summary
# ------------------------------
block_summary_df = (
    importance_df.groupby("feature_block", as_index=False)
    .agg(
        n_features=("feature_name_original", "count"),
        mean_importance_score_ibs=("importance_score_ibs", "mean"),
        median_importance_score_ibs=("importance_score_ibs", "median"),
        max_importance_score_ibs=("importance_score_ibs", "max"),
        mean_importance_score_auc=("mean_importance_score_auc", "mean"),
    )
    .sort_values(by="mean_importance_score_ibs", ascending=False)
    .reset_index(drop=True)
)

top_features_df = importance_sorted_df.head(20).reset_index(drop=True)

# ------------------------------
# 11) Save outputs
# ------------------------------
feature_table_path = TABLES_DIR / "table_neural_explainability_grouped_permutation_importance.csv"
block_summary_path = TABLES_DIR / "table_neural_explainability_grouped_block_summary.csv"
top_features_path = TABLES_DIR / "table_neural_explainability_grouped_top_features.csv"
config_path = METADATA_DIR / "neural_explainability_grouped_summary.json"

importance_sorted_df.to_csv(feature_table_path, index=False)
block_summary_df.to_csv(block_summary_path, index=False)
top_features_df.to_csv(top_features_path, index=False)

save_json(
    {
        "model_id": "neural_tuned",
        "preprocessor_file_used": str(preprocessor_path),
        "train_data_used": str(train_data_path),
        "test_data_used": str(test_data_path),
        "n_feature_groups": int(len(FEATURE_GROUPS)),
        "baseline_ibs_refit_model": float(baseline_ibs) if pd.notna(baseline_ibs) else None,
        "top_feature_by_grouped_permutation_ibs": top_features_df.iloc[0]["feature_name_original"],
        "top_feature_block_by_mean_importance_ibs": block_summary_df.iloc[0]["feature_block"],
    },
    config_path,
)

# ------------------------------
# 12) Output for feedback
# ------------------------------
print("\nTop neural features by GROUPED permutation importance:")
display(top_features_df)

print("\nNeural grouped feature-block summary:")
display(block_summary_df)

print("\nSaved:")
print("-", feature_table_path.resolve())
print("-", block_summary_path.resolve())
print("-", top_features_path.resolve())
print("-", config_path.resolve())


# G4 — Explainability for Cox Tuned (Revised)

In [ ]:
# ==============================================================
# G4 — Explainability for Cox Tuned (Revised)
# --------------------------------------------------------------
# Purpose:
#   Produce global explainability outputs for the tuned Cox
#   comparable benchmark.
#
# Methodological note:
#   This step uses direct parameter interpretation because the
#   tuned Cox model is intrinsically interpretable.
# ==============================================================

print("\n" + "=" * 70)
print("G4 — Explainability for Cox Tuned (Revised)")
print("=" * 70)
print("Methodological note: this step computes global explainability")
print("for the tuned Cox comparable benchmark.")

# ------------------------------
# 1) Basic checks
# ------------------------------
required_names = ["OUTPUT_DIR", "TABLES_DIR", "METADATA_DIR", "save_json"]
missing_names = [name for name in required_names if name not in globals()]
if missing_names:
    raise NameError(
        "Missing required objects from previous cells: "
        + ", ".join(missing_names)
        + ". Please run prior setup cells first."
    )

from pathlib import Path
import numpy as np
import pandas as pd
import joblib

# ------------------------------
# 2) Paths
# ------------------------------
MODEL_DIR = OUTPUT_DIR / "models"

model_path = MODEL_DIR / "cox_early_window_tuned.joblib"
if not model_path.exists():
    raise FileNotFoundError(f"Tuned model file not found: {model_path}")

preprocessor_path = MODEL_DIR / "cox_preprocessor.joblib"

if not model_path.exists():
    raise FileNotFoundError(f"Model file not found: {model_path}")
if not preprocessor_path.exists():
    raise FileNotFoundError(f"Preprocessor file not found: {preprocessor_path}")

# ------------------------------
# 3) Load artifacts
# ------------------------------
cox_model = joblib.load(model_path)
cox_preprocessor = joblib.load(preprocessor_path)

# ------------------------------
# 4) Recover transformed feature names
# ------------------------------
if not hasattr(cox_preprocessor, "get_feature_names_out"):
    raise AttributeError("The loaded preprocessor does not expose get_feature_names_out().")

feature_names_out = list(cox_preprocessor.get_feature_names_out())

if not hasattr(cox_model, "params_"):
    raise AttributeError("Loaded Cox model does not expose params_.")

coef_series = cox_model.params_.copy()
param_names = coef_series.index.tolist()


### Funcao: map_param_name

Definicao isolada de map_param_name para reutilizacao nas etapas seguintes.


In [ ]:

def map_param_name(name, feature_names_out):
    if isinstance(name, str) and name.startswith("x"):
        suffix = name[1:]
        if suffix.isdigit():
            idx = int(suffix)
            if idx < len(feature_names_out):
                return feature_names_out[idx]
    return name


In [ ]:

mapped_feature_names = [map_param_name(name, feature_names_out) for name in param_names]

# ------------------------------
# 5) Robustly load summary and rename first column
# ------------------------------
summary_df = cox_model.summary.copy().reset_index()

if summary_df.shape[1] == 0:
    raise ValueError("Cox model summary is empty after reset_index().")

first_col_name = summary_df.columns[0]
summary_df = summary_df.rename(columns={first_col_name: "raw_feature_name"})

if "coef" not in summary_df.columns:
    raise ValueError("Cox model summary does not contain 'coef' column.")

raw_feature_names_summary = summary_df["raw_feature_name"].tolist()
mapped_summary_names = [map_param_name(name, feature_names_out) for name in raw_feature_names_summary]
summary_df["feature_name_out"] = mapped_summary_names

# ------------------------------
# 6) Build explainability table
# ------------------------------
keep_cols = [
    "feature_name_out",
    "coef",
    "exp(coef)",
    "se(coef)",
    "coef lower 95%",
    "coef upper 95%",
    "exp(coef) lower 95%",
    "exp(coef) upper 95%",
    "z",
    "p",
]

available_keep_cols = [c for c in keep_cols if c in summary_df.columns]
explain_df = summary_df[available_keep_cols].copy()

rename_map = {
    "coef": "coefficient",
    "exp(coef)": "hazard_ratio",
    "se(coef)": "se_coefficient",
    "coef lower 95%": "coef_lower_95",
    "coef upper 95%": "coef_upper_95",
    "exp(coef) lower 95%": "hazard_ratio_lower_95",
    "exp(coef) upper 95%": "hazard_ratio_upper_95",
    "p": "p_value",
}
explain_df.rename(columns=rename_map, inplace=True)

explain_df["abs_coefficient"] = explain_df["coefficient"].abs()


### Funcao: infer_block

Definicao isolada de infer_block para reutilizacao nas etapas seguintes.


In [ ]:

def infer_block(feature_name: str) -> str:
    if str(feature_name).startswith("num__clicks_first_4_weeks"):
        return "early_window_behavior"
    if str(feature_name).startswith("num__active_weeks_first_4"):
        return "early_window_behavior"
    if str(feature_name).startswith("num__mean_clicks_first_4_weeks"):
        return "early_window_behavior"
    if str(feature_name).startswith("num__num_of_prev_attempts"):
        return "static_structural"
    if str(feature_name).startswith("num__studied_credits"):
        return "static_structural"
    if str(feature_name).startswith("cat__gender_"):
        return "static_structural"
    if str(feature_name).startswith("cat__region_"):
        return "static_structural"
    if str(feature_name).startswith("cat__highest_education_"):
        return "static_structural"
    if str(feature_name).startswith("cat__imd_band_"):
        return "static_structural"
    if str(feature_name).startswith("cat__age_band_"):
        return "static_structural"
    if str(feature_name).startswith("cat__disability_"):
        return "static_structural"
    return "other"


In [ ]:

explain_df["feature_block"] = explain_df["feature_name_out"].apply(infer_block)

explain_df["effect_direction"] = np.where(
    explain_df["coefficient"] > 0,
    "increases_hazard",
    np.where(
        explain_df["coefficient"] < 0,
        "decreases_hazard",
        "neutral"
    )
)

explain_df_sorted_abs = explain_df.sort_values(
    by="abs_coefficient", ascending=False
).reset_index(drop=True)

explain_df_sorted_signed = explain_df.sort_values(
    by="coefficient", ascending=False
).reset_index(drop=True)

# ------------------------------
# 7) Block-level summary
# ------------------------------
block_summary_df = (
    explain_df.groupby("feature_block", as_index=False)
    .agg(
        n_features=("feature_name_out", "count"),
        mean_abs_coefficient=("abs_coefficient", "mean"),
        median_abs_coefficient=("abs_coefficient", "median"),
        max_abs_coefficient=("abs_coefficient", "max"),
        mean_coefficient=("coefficient", "mean"),
        mean_hazard_ratio=("hazard_ratio", "mean"),
    )
    .sort_values(by="mean_abs_coefficient", ascending=False)
    .reset_index(drop=True)
)

# ------------------------------
# 8) Top positive / negative effects
# ------------------------------
top_positive_df = explain_df.sort_values(
    by="coefficient", ascending=False
).head(15).reset_index(drop=True)

top_negative_df = explain_df.sort_values(
    by="coefficient", ascending=True
).head(15).reset_index(drop=True)

# ------------------------------
# 9) Significant effects only
# ------------------------------
if "p_value" in explain_df.columns:
    significant_df = explain_df[explain_df["p_value"] < 0.05].copy()
    significant_df = significant_df.sort_values(
        by="abs_coefficient", ascending=False
    ).reset_index(drop=True)
else:
    significant_df = explain_df.iloc[0:0].copy()

# ------------------------------
# 10) Save outputs
# ------------------------------
feature_table_path = TABLES_DIR / "table_cox_explainability_feature_coefficients.csv"
signed_table_path = TABLES_DIR / "table_cox_explainability_signed_effects.csv"
block_summary_path = TABLES_DIR / "table_cox_explainability_block_summary.csv"
top_positive_path = TABLES_DIR / "table_cox_explainability_top_positive.csv"
top_negative_path = TABLES_DIR / "table_cox_explainability_top_negative.csv"
significant_path = TABLES_DIR / "table_cox_explainability_significant_effects.csv"
config_path = METADATA_DIR / "cox_explainability_summary.json"

explain_df_sorted_abs.to_csv(feature_table_path, index=False)
explain_df_sorted_signed.to_csv(signed_table_path, index=False)
block_summary_df.to_csv(block_summary_path, index=False)
top_positive_df.to_csv(top_positive_path, index=False)
top_negative_df.to_csv(top_negative_path, index=False)
significant_df.to_csv(significant_path, index=False)

save_json(
    {
        "model_id": "cox_tuned",
        "model_file_used": str(model_path),
        "preprocessor_file_used": str(preprocessor_path),
        "n_transformed_features_in_summary": int(explain_df.shape[0]),
        "top_feature_by_abs_coef": explain_df_sorted_abs.iloc[0]["feature_name_out"],
        "top_feature_block_by_mean_abs_coef": block_summary_df.iloc[0]["feature_block"],
        "n_significant_features_p_lt_0_05": int(significant_df.shape[0]),
    },
    config_path,
)

# ------------------------------
# 11) Output for feedback
# ------------------------------
print("\nTop Cox features by absolute coefficient:")
display(explain_df_sorted_abs.head(20))

print("\nTop positive Cox effects:")
display(top_positive_df)

print("\nTop negative Cox effects:")
display(top_negative_df)

print("\nSignificant Cox effects (p < 0.05):")
display(significant_df.head(20))

print("\nCox feature-block summary:")
display(block_summary_df)

print("\nSaved:")
print("-", feature_table_path.resolve())
print("-", signed_table_path.resolve())
print("-", block_summary_path.resolve())
print("-", top_positive_path.resolve())
print("-", top_negative_path.resolve())
print("-", significant_path.resolve())
print("-", config_path.resolve())


# G5 — Explainability for DeepSurv Tuned

In [ ]:
# ==============================================================
# G5 — Explainability for DeepSurv Tuned
# --------------------------------------------------------------
# Purpose:
#   Produce global explainability outputs for the tuned DeepSurv
#   model using GROUPED permutation importance at the original-
#   feature level.
#
# Methodological note:
#   This step follows the same corrected explainability logic
#   used for the tuned neural model:
#   - permutation is applied by original feature/group
#   - not by individual one-hot encoded transformed column
#
# Scope:
#   Enrollment-level continuous-time DeepSurv benchmark.
# ==============================================================

print("\n" + "=" * 70)
print("G5 — Explainability for DeepSurv Tuned")
print("=" * 70)
print("Methodological note: this step computes global explainability")
print("for the tuned DeepSurv model using GROUPED permutation")
print("importance at the original-feature level.")

# ------------------------------
# 1) Basic checks
# ------------------------------
required_names = [
    "OUTPUT_DIR", "TABLES_DIR", "METADATA_DIR",
    "RANDOM_SEED", "HORIZONS_WEEKS", "save_json"
]
missing_names = [name for name in required_names if name not in globals()]
if missing_names:
    raise NameError(
        "Missing required objects from previous cells: "
        + ", ".join(missing_names)
        + ". Please run prior setup cells first."
    )

from pathlib import Path
import numpy as np
import pandas as pd
import scipy
import torch
import torchtuples as tt
import joblib

from sklearn.metrics import roc_auc_score

try:
    from pycox.evaluation import EvalSurv
    PYCOX_AVAILABLE = True
except Exception:
    PYCOX_AVAILABLE = False

if not PYCOX_AVAILABLE:
    raise ImportError("pycox is required for P36.")

# ------------------------------
# 2) Compatibility patch for SciPy / PyCox
# ------------------------------
try:
    if not hasattr(scipy.integrate, "simps") and hasattr(scipy.integrate, "simpson"):
        def _simps_compat(y, x=None, dx=1.0, axis=-1, even=None):
            return scipy.integrate.simpson(y, x=x, dx=dx, axis=axis)
        scipy.integrate.simps = _simps_compat
except Exception:
    pass

# ------------------------------
# 3) Paths
# ------------------------------
MODEL_DIR = OUTPUT_DIR / "models"
DATA_DIR = OUTPUT_DIR / "data"

preprocessor_path = MODEL_DIR / "deepsurv_preprocessor.joblib"
train_data_path = DATA_DIR / "enrollment_deepsurv_ready_train.parquet"
test_data_path = DATA_DIR / "enrollment_deepsurv_ready_test.parquet"

if not train_data_path.exists():
    train_data_path = DATA_DIR / "enrollment_deepsurv_ready_train.csv"
if not test_data_path.exists():
    test_data_path = DATA_DIR / "enrollment_deepsurv_ready_test.csv"

if not preprocessor_path.exists():
    raise FileNotFoundError(f"Preprocessor file not found: {preprocessor_path}")
if not train_data_path.exists():
    raise FileNotFoundError(f"Train data file not found: {train_data_path}")
if not test_data_path.exists():
    raise FileNotFoundError(f"Test data file not found: {test_data_path}")

# ------------------------------
# 4) Load artifacts
# ------------------------------
deepsurv_preprocessor = joblib.load(preprocessor_path)

if str(train_data_path).endswith(".parquet"):
    deepsurv_train_df = pd.read_parquet(train_data_path)
else:
    deepsurv_train_df = pd.read_csv(train_data_path)

if str(test_data_path).endswith(".parquet"):
    deepsurv_test_df = pd.read_parquet(test_data_path)
else:
    deepsurv_test_df = pd.read_csv(test_data_path)

# ------------------------------
# 5) Column definitions
# ------------------------------
AUX_CONTINUOUS = [
    "enrollment_id",
    "id_student",
    "code_module",
    "code_presentation",
    "split",
    "time_for_split",
    "time_bucket",
    "event_time_bucket_label",
]

TARGET_EVENT_COL = "event"
TARGET_DURATION_COL = "duration"

FEATURE_GROUPS = [
    "gender",
    "region",
    "highest_education",
    "imd_band",
    "age_band",
    "disability",
    "num_of_prev_attempts",
    "studied_credits",
    "clicks_first_4_weeks",
    "active_weeks_first_4",
    "mean_clicks_first_4_weeks",
]


### Funcao: get_feature_columns

Definicao isolada de get_feature_columns para reutilizacao nas etapas seguintes.


In [ ]:

def get_feature_columns(df: pd.DataFrame):
    excluded = set(AUX_CONTINUOUS + [TARGET_EVENT_COL, TARGET_DURATION_COL, "duration_raw", "used_zero_week_fallback_for_censoring"])
    return [c for c in df.columns if c not in excluded]


In [ ]:

feature_cols_raw = get_feature_columns(deepsurv_train_df)

missing_expected_features = [c for c in FEATURE_GROUPS if c not in feature_cols_raw]
if missing_expected_features:
    raise ValueError(
        f"Missing expected feature groups in DeepSurv data: {missing_expected_features}"
    )

# transformed design
X_train = deepsurv_preprocessor.transform(deepsurv_train_df[feature_cols_raw])
X_test = deepsurv_preprocessor.transform(deepsurv_test_df[feature_cols_raw])

if hasattr(X_train, "toarray"):
    X_train_dense = X_train.toarray().astype(np.float32)
    X_test_dense = X_test.toarray().astype(np.float32)
else:
    X_train_dense = np.asarray(X_train).astype(np.float32)
    X_test_dense = np.asarray(X_test).astype(np.float32)

duration_train = deepsurv_train_df[TARGET_DURATION_COL].to_numpy().astype(np.float32)
event_train = deepsurv_train_df[TARGET_EVENT_COL].to_numpy().astype(np.float32)

duration_test = deepsurv_test_df[TARGET_DURATION_COL].to_numpy().astype(np.float32)
event_test = deepsurv_test_df[TARGET_EVENT_COL].to_numpy().astype(np.float32)

# ------------------------------
# 6) Refit tuned DeepSurv model
# ------------------------------
# Based on tuned winner family size from P25:
# hidden_dims=[64, 32], dropout=0.3, lr=5e-4, weight_decay=1e-4

torch.manual_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

net = tt.practical.MLPVanilla(
    in_features=X_train_dense.shape[1],
    num_nodes=[64, 32],
    out_features=1,
    batch_norm=True,
    dropout=0.3,
    output_bias=False,
)

model = tt.Model(net, torch.nn.BCEWithLogitsLoss(), tt.optim.AdamW)
model.optimizer.set_lr(5e-4)

# NOTE:
# This is a practical benchmark-side refit for explainability.
# We keep the same tuned architecture family and optimizer scale.
_ = model.fit(
    X_train_dense,
    event_train.reshape(-1, 1),
    batch_size=256,
    epochs=55,
    verbose=False,
)


### Funcao: build_survival_df_from_risk_scores

Approximate survival curves from relative risk scores using a


In [ ]:

# ------------------------------
# 7) Survival helper
# ------------------------------
def build_survival_df_from_risk_scores(
    risk_scores: np.ndarray,
    train_duration: np.ndarray,
    train_event: np.ndarray,
    test_duration: np.ndarray
) -> pd.DataFrame:
    """
    Approximate survival curves from relative risk scores using a
    Breslow-style baseline hazard estimated from the TRAIN set.
    """

    risk_scores = np.asarray(risk_scores).reshape(-1)
    train_duration_i = np.asarray(train_duration).astype(int)
    train_event_i = np.asarray(train_event).astype(int)
    test_duration_i = np.asarray(test_duration).astype(int)

    # approximate train scores from current model
    train_logits = model.predict(X_train_dense.astype(np.float32)).reshape(-1)
    train_risk_scores = np.exp(train_logits)

    unique_event_times = np.sort(np.unique(train_duration_i[train_event_i == 1]))
    if len(unique_event_times) == 0:
        raise ValueError("No event times found in training set for DeepSurv baseline estimation.")

    baseline_hazard = []
    for t in unique_event_times:
        d_t = np.sum((train_duration_i == t) & (train_event_i == 1))
        at_risk = train_duration_i >= t
        denom = np.sum(train_risk_scores[at_risk])
        h0_t = d_t / denom if denom > 0 else 0.0
        baseline_hazard.append(h0_t)

    baseline_hazard = np.asarray(baseline_hazard, dtype=float)
    baseline_cum_hazard = np.cumsum(baseline_hazard)

    max_t = int(max(np.max(test_duration_i), np.max(unique_event_times)))
    full_times = np.arange(0, max_t + 1, dtype=int)

    cumhaz_full = np.zeros_like(full_times, dtype=float)
    time_to_ch = {int(t): ch for t, ch in zip(unique_event_times, baseline_cum_hazard)}
    running = 0.0
    for i, t in enumerate(full_times):
        if t in time_to_ch:
            running = time_to_ch[t]
        cumhaz_full[i] = running

    surv_cols = {}
    for i in range(len(risk_scores)):
        rs = np.exp(risk_scores[i])
        surv_cols[i] = np.exp(-cumhaz_full * rs)

    surv_df = pd.DataFrame(surv_cols, index=full_times)
    surv_df.index.name = "time"
    return surv_df


### Funcao: get_surv_at_horizon

Definicao isolada de get_surv_at_horizon para reutilizacao nas etapas seguintes.


In [ ]:

def get_surv_at_horizon(surv_frame: pd.DataFrame, h: int) -> pd.Series:
    idx = np.asarray(surv_frame.index, dtype=float)
    pos = np.searchsorted(idx, float(h), side="right") - 1
    if pos < 0:
        return pd.Series(np.ones(surv_frame.shape[1]), index=surv_frame.columns)
    return surv_frame.iloc[pos]


### Funcao: evaluate_continuous_survival_from_risk_scores

Definicao isolada de evaluate_continuous_survival_from_risk_scores para reutilizacao nas etapas seguintes.


In [ ]:

def evaluate_continuous_survival_from_risk_scores(
    risk_scores_test: np.ndarray,
    duration_test: np.ndarray,
    event_test: np.ndarray,
    horizons: list[int]
):
    surv_df = build_survival_df_from_risk_scores(
        risk_scores=risk_scores_test,
        train_duration=duration_train,
        train_event=event_train,
        test_duration=duration_test,
    )

    eval_surv = EvalSurv(
        surv=surv_df,
        durations=duration_test.astype(int),
        events=event_test.astype(int),
        censor_surv="km",
    )

    try:
        max_requested_horizon = int(max(horizons))
        ibs_grid = np.arange(1, max_requested_horizon + 1, dtype=int)
        ibs_value = float(eval_surv.integrated_brier_score(ibs_grid))
    except Exception:
        ibs_value = np.nan

    truth_test = pd.DataFrame({
        "duration": duration_test.astype(int),
        "event": event_test.astype(int),
    })

    risk_auc_rows = []
    for h in horizons:
        pred_surv_h = get_surv_at_horizon(surv_df, h)
        pred_risk_h = 1.0 - pred_surv_h

        eval_df = truth_test.copy()
        eval_df["pred_risk_h"] = pred_risk_h.values

        eval_df["is_evaluable_at_h"] = (
            ((eval_df["event"] == 1) & (eval_df["duration"] <= h)) |
            (eval_df["duration"] >= h)
        ).astype(int)

        eval_df = eval_df[eval_df["is_evaluable_at_h"] == 1].copy()
        eval_df["observed_event_by_h"] = ((eval_df["event"] == 1) & (eval_df["duration"] <= h)).astype(int)

        if eval_df["observed_event_by_h"].nunique() >= 2:
            risk_auc = roc_auc_score(eval_df["observed_event_by_h"], eval_df["pred_risk_h"])
        else:
            risk_auc = np.nan

        risk_auc_rows.append({
            "horizon_week": h,
            "risk_auc_at_horizon": float(risk_auc) if pd.notna(risk_auc) else np.nan,
        })

    return {
        "ibs": ibs_value,
        "risk_auc_df": pd.DataFrame(risk_auc_rows),
        "surv_df": surv_df,
    }


### Funcao: predict_risk_score_from_raw_df

Definicao isolada de predict_risk_score_from_raw_df para reutilizacao nas etapas seguintes.


In [ ]:

def predict_risk_score_from_raw_df(raw_df: pd.DataFrame) -> np.ndarray:
    X = deepsurv_preprocessor.transform(raw_df[feature_cols_raw])
    if hasattr(X, "toarray"):
        X_dense = X.toarray().astype(np.float32)
    else:
        X_dense = np.asarray(X).astype(np.float32)

    logits = model.predict(X_dense).reshape(-1)
    return logits


### Funcao: infer_block

Definicao isolada de infer_block para reutilizacao nas etapas seguintes.


In [ ]:

def infer_block(feature_name: str) -> str:
    if feature_name in [
        "clicks_first_4_weeks",
        "active_weeks_first_4",
        "mean_clicks_first_4_weeks",
    ]:
        return "early_window_behavior"
    return "static_structural"


In [ ]:

# ------------------------------
# 8) Baseline performance
# ------------------------------
baseline_risk_scores = predict_risk_score_from_raw_df(deepsurv_test_df)

baseline_eval = evaluate_continuous_survival_from_risk_scores(
    risk_scores_test=baseline_risk_scores,
    duration_test=duration_test,
    event_test=event_test,
    horizons=HORIZONS_WEEKS,
)

baseline_ibs = baseline_eval["ibs"]
baseline_risk_auc_df = baseline_eval["risk_auc_df"].copy()

baseline_risk_auc_h10 = float(
    baseline_risk_auc_df.loc[baseline_risk_auc_df["horizon_week"] == 10, "risk_auc_at_horizon"].iloc[0]
)
baseline_risk_auc_h20 = float(
    baseline_risk_auc_df.loc[baseline_risk_auc_df["horizon_week"] == 20, "risk_auc_at_horizon"].iloc[0]
)
baseline_risk_auc_h30 = float(
    baseline_risk_auc_df.loc[baseline_risk_auc_df["horizon_week"] == 30, "risk_auc_at_horizon"].iloc[0]
)

# ------------------------------
# 9) GROUPED permutation importance
# ------------------------------
rng = np.random.default_rng(RANDOM_SEED)
importance_rows = []

for feat in FEATURE_GROUPS:
    perm_df = deepsurv_test_df.copy()

    shuffled = perm_df[feat].to_numpy(copy=True)
    rng.shuffle(shuffled)
    perm_df[feat] = shuffled

    perm_risk_scores = predict_risk_score_from_raw_df(perm_df)

    perm_eval = evaluate_continuous_survival_from_risk_scores(
        risk_scores_test=perm_risk_scores,
        duration_test=duration_test,
        event_test=event_test,
        horizons=HORIZONS_WEEKS,
    )

    perm_ibs = perm_eval["ibs"]
    perm_risk_auc_df = perm_eval["risk_auc_df"].copy()

    perm_risk_auc_h10 = float(
        perm_risk_auc_df.loc[perm_risk_auc_df["horizon_week"] == 10, "risk_auc_at_horizon"].iloc[0]
    )
    perm_risk_auc_h20 = float(
        perm_risk_auc_df.loc[perm_risk_auc_df["horizon_week"] == 20, "risk_auc_at_horizon"].iloc[0]
    )
    perm_risk_auc_h30 = float(
        perm_risk_auc_df.loc[perm_risk_auc_df["horizon_week"] == 30, "risk_auc_at_horizon"].iloc[0]
    )

    importance_rows.append({
        "feature_name_original": feat,
        "baseline_ibs": baseline_ibs,
        "permuted_ibs": perm_ibs,
        "delta_ibs": perm_ibs - baseline_ibs if pd.notna(perm_ibs) and pd.notna(baseline_ibs) else np.nan,
        "baseline_risk_auc_h10": baseline_risk_auc_h10,
        "permuted_risk_auc_h10": perm_risk_auc_h10,
        "delta_risk_auc_h10": perm_risk_auc_h10 - baseline_risk_auc_h10 if pd.notna(perm_risk_auc_h10) else np.nan,
        "baseline_risk_auc_h20": baseline_risk_auc_h20,
        "permuted_risk_auc_h20": perm_risk_auc_h20,
        "delta_risk_auc_h20": perm_risk_auc_h20 - baseline_risk_auc_h20 if pd.notna(perm_risk_auc_h20) else np.nan,
        "baseline_risk_auc_h30": baseline_risk_auc_h30,
        "permuted_risk_auc_h30": perm_risk_auc_h30,
        "delta_risk_auc_h30": perm_risk_auc_h30 - baseline_risk_auc_h30 if pd.notna(perm_risk_auc_h30) else np.nan,
    })

importance_df = pd.DataFrame(importance_rows)
importance_df["importance_score_ibs"] = importance_df["delta_ibs"]
importance_df["importance_score_risk_auc_h10"] = -importance_df["delta_risk_auc_h10"]
importance_df["importance_score_risk_auc_h20"] = -importance_df["delta_risk_auc_h20"]
importance_df["importance_score_risk_auc_h30"] = -importance_df["delta_risk_auc_h30"]
importance_df["mean_importance_score_auc"] = importance_df[
    ["importance_score_risk_auc_h10", "importance_score_risk_auc_h20", "importance_score_risk_auc_h30"]
].mean(axis=1)
importance_df["feature_block"] = importance_df["feature_name_original"].apply(infer_block)

importance_sorted_df = importance_df.sort_values(
    by=["importance_score_ibs", "mean_importance_score_auc"],
    ascending=[False, False]
).reset_index(drop=True)

# ------------------------------
# 10) Block summary
# ------------------------------
block_summary_df = (
    importance_df.groupby("feature_block", as_index=False)
    .agg(
        n_features=("feature_name_original", "count"),
        mean_importance_score_ibs=("importance_score_ibs", "mean"),
        median_importance_score_ibs=("importance_score_ibs", "median"),
        max_importance_score_ibs=("importance_score_ibs", "max"),
        mean_importance_score_auc=("mean_importance_score_auc", "mean"),
    )
    .sort_values(by="mean_importance_score_ibs", ascending=False)
    .reset_index(drop=True)
)

top_features_df = importance_sorted_df.head(20).reset_index(drop=True)

# ------------------------------
# 11) Save outputs
# ------------------------------
feature_table_path = TABLES_DIR / "table_deepsurv_explainability_grouped_permutation_importance.csv"
block_summary_path = TABLES_DIR / "table_deepsurv_explainability_grouped_block_summary.csv"
top_features_path = TABLES_DIR / "table_deepsurv_explainability_grouped_top_features.csv"
config_path = METADATA_DIR / "deepsurv_explainability_grouped_summary.json"

importance_sorted_df.to_csv(feature_table_path, index=False)
block_summary_df.to_csv(block_summary_path, index=False)
top_features_df.to_csv(top_features_path, index=False)

save_json(
    {
        "model_id": "deepsurv_tuned",
        "preprocessor_file_used": str(preprocessor_path),
        "train_data_used": str(train_data_path),
        "test_data_used": str(test_data_path),
        "n_feature_groups": int(len(FEATURE_GROUPS)),
        "baseline_ibs_refit_model": float(baseline_ibs) if pd.notna(baseline_ibs) else None,
        "top_feature_by_grouped_permutation_ibs": top_features_df.iloc[0]["feature_name_original"],
        "top_feature_block_by_mean_importance_ibs": block_summary_df.iloc[0]["feature_block"],
    },
    config_path,
)

# ------------------------------
# 12) Output for feedback
# ------------------------------
print("\nTop DeepSurv features by GROUPED permutation importance:")
display(top_features_df)

print("\nDeepSurv grouped feature-block summary:")
display(block_summary_df)

print("\nSaved:")
print("-", feature_table_path.resolve())
print("-", block_summary_path.resolve())
print("-", top_features_path.resolve())
print("-", config_path.resolve())


# G6 — Consolidate Explainability Across All Tuned Families

### Funcao: safe_top_feature

Definicao isolada de safe_top_feature para reutilizacao nas etapas seguintes.


In [ ]:
# ==============================================================
# G6 — Consolidate Explainability Across All Tuned Families
# --------------------------------------------------------------
# Purpose:
#   Consolidate explainability outputs across all tuned model
#   families into cross-family comparison tables.
#
# Inputs expected from previous cells:
#   - OUTPUT_DIR
#   - TABLES_DIR
#   - METADATA_DIR
#   - save_json
#
# Expected existing outputs:
#   - Linear tuned explainability tables (G2)
#   - Neural tuned grouped explainability tables (G3 revised v2)
#   - Cox tuned explainability tables (G4)
#   - DeepSurv tuned grouped explainability tables (G5)
#
# Outputs:
#   - consolidated explainability summary by model
#   - cross-family feature-block comparison
#   - top drivers by model
#   - convergence summary across families
# ==============================================================

print("\n" + "=" * 70)
print("G6 — Consolidate Explainability Across All Tuned Families")
print("=" * 70)
print("Methodological note: this step consolidates explainability outputs only.")
print("No model is trained and no new explanation is computed here.")

# ------------------------------
# 1) Basic checks
# ------------------------------
required_names = ["OUTPUT_DIR", "TABLES_DIR", "METADATA_DIR", "save_json"]
missing_names = [name for name in required_names if name not in globals()]
if missing_names:
    raise NameError(
        "Missing required objects from previous cells: "
        + ", ".join(missing_names)
        + ". Please run prior setup cells first."
    )

from pathlib import Path
import numpy as np
import pandas as pd

# ------------------------------
# 2) Paths
# ------------------------------
# Linear
linear_feature_path = TABLES_DIR / "table_linear_explainability_feature_coefficients.csv"
linear_block_path = TABLES_DIR / "table_linear_explainability_block_summary.csv"

# Neural
neural_feature_path = TABLES_DIR / "table_neural_explainability_grouped_permutation_importance.csv"
neural_block_path = TABLES_DIR / "table_neural_explainability_grouped_block_summary.csv"

# Cox
cox_feature_path = TABLES_DIR / "table_cox_explainability_feature_coefficients.csv"
cox_block_path = TABLES_DIR / "table_cox_explainability_block_summary.csv"

# DeepSurv
deepsurv_feature_path = TABLES_DIR / "table_deepsurv_explainability_grouped_permutation_importance.csv"
deepsurv_block_path = TABLES_DIR / "table_deepsurv_explainability_grouped_block_summary.csv"

required_paths = [
    linear_feature_path, linear_block_path,
    neural_feature_path, neural_block_path,
    cox_feature_path, cox_block_path,
    deepsurv_feature_path, deepsurv_block_path,
]
missing_paths = [str(p) for p in required_paths if not p.exists()]
if missing_paths:
    raise FileNotFoundError(
        "Missing explainability input files:\n- " + "\n- ".join(missing_paths)
    )

# ------------------------------
# 3) Load tables
# ------------------------------
linear_feature_df = pd.read_csv(linear_feature_path)
linear_block_df = pd.read_csv(linear_block_path)

neural_feature_df = pd.read_csv(neural_feature_path)
neural_block_df = pd.read_csv(neural_block_path)

cox_feature_df = pd.read_csv(cox_feature_path)
cox_block_df = pd.read_csv(cox_block_path)

deepsurv_feature_df = pd.read_csv(deepsurv_feature_path)
deepsurv_block_df = pd.read_csv(deepsurv_block_path)

# ------------------------------
# 4) Normalize feature-level tables
# ------------------------------
# Linear / Cox are coefficient-based
linear_norm = linear_feature_df.copy()
linear_norm["model_id"] = "linear_tuned"
linear_norm["display_name"] = "Linear Discrete-Time Hazard (Tuned)"
linear_norm["family"] = "discrete_time_linear"
linear_norm["explainability_type"] = "coefficient_based"
linear_norm["feature_name"] = linear_norm["feature_name_out"]
linear_norm["importance_primary"] = linear_norm["abs_coefficient"]
linear_norm["importance_secondary"] = linear_norm["coefficient"]
linear_norm["importance_primary_label"] = "abs_coefficient"
linear_norm["importance_secondary_label"] = "signed_coefficient"

cox_norm = cox_feature_df.copy()
cox_norm["model_id"] = "cox_tuned"
cox_norm["display_name"] = "Cox Comparable (Tuned)"
cox_norm["family"] = "continuous_time_cox"
cox_norm["explainability_type"] = "coefficient_based"
cox_norm["feature_name"] = cox_norm["feature_name_out"]
cox_norm["importance_primary"] = cox_norm["abs_coefficient"]
cox_norm["importance_secondary"] = cox_norm["coefficient"]
cox_norm["importance_primary_label"] = "abs_coefficient"
cox_norm["importance_secondary_label"] = "signed_coefficient"

# Neural / DeepSurv are grouped permutation-based
neural_norm = neural_feature_df.copy()
neural_norm["model_id"] = "neural_tuned"
neural_norm["display_name"] = "Neural Discrete-Time Survival (Tuned)"
neural_norm["family"] = "discrete_time_neural"
neural_norm["explainability_type"] = "grouped_permutation"
neural_norm["feature_name"] = neural_norm["feature_name_original"]
neural_norm["importance_primary"] = neural_norm["importance_score_ibs"]
neural_norm["importance_secondary"] = neural_norm["mean_importance_score_auc"]
neural_norm["importance_primary_label"] = "delta_ibs_after_grouped_permutation"
neural_norm["importance_secondary_label"] = "mean_auc_importance_after_grouped_permutation"

deepsurv_norm = deepsurv_feature_df.copy()
deepsurv_norm["model_id"] = "deepsurv_tuned"
deepsurv_norm["display_name"] = "DeepSurv (Tuned)"
deepsurv_norm["family"] = "continuous_time_deepsurv"
deepsurv_norm["explainability_type"] = "grouped_permutation"
deepsurv_norm["feature_name"] = deepsurv_norm["feature_name_original"]
deepsurv_norm["importance_primary"] = deepsurv_norm["importance_score_ibs"]
deepsurv_norm["importance_secondary"] = deepsurv_norm["mean_importance_score_auc"]
deepsurv_norm["importance_primary_label"] = "delta_ibs_after_grouped_permutation"
deepsurv_norm["importance_secondary_label"] = "mean_auc_importance_after_grouped_permutation"

feature_compare_cols = [
    "model_id", "display_name", "family", "explainability_type",
    "feature_name", "feature_block",
    "importance_primary", "importance_secondary",
    "importance_primary_label", "importance_secondary_label"
]

all_features_df = pd.concat([
    linear_norm[feature_compare_cols],
    neural_norm[feature_compare_cols],
    cox_norm[feature_compare_cols],
    deepsurv_norm[feature_compare_cols],
], ignore_index=True)

# ------------------------------
# 5) Normalize block-level tables
# ------------------------------
linear_block_norm = linear_block_df.copy()
linear_block_norm["model_id"] = "linear_tuned"
linear_block_norm["display_name"] = "Linear Discrete-Time Hazard (Tuned)"
linear_block_norm["family"] = "discrete_time_linear"
linear_block_norm["block_importance_primary"] = linear_block_norm["mean_abs_coefficient"]
linear_block_norm["block_importance_secondary"] = linear_block_norm["mean_coefficient"]
linear_block_norm["block_primary_label"] = "mean_abs_coefficient"
linear_block_norm["block_secondary_label"] = "mean_signed_coefficient"

neural_block_norm = neural_block_df.copy()
neural_block_norm["model_id"] = "neural_tuned"
neural_block_norm["display_name"] = "Neural Discrete-Time Survival (Tuned)"
neural_block_norm["family"] = "discrete_time_neural"
neural_block_norm["block_importance_primary"] = neural_block_norm["mean_importance_score_ibs"]
neural_block_norm["block_importance_secondary"] = neural_block_norm["mean_importance_score_auc"]
neural_block_norm["block_primary_label"] = "mean_delta_ibs_after_grouped_permutation"
neural_block_norm["block_secondary_label"] = "mean_auc_importance_after_grouped_permutation"

cox_block_norm = cox_block_df.copy()
cox_block_norm["model_id"] = "cox_tuned"
cox_block_norm["display_name"] = "Cox Comparable (Tuned)"
cox_block_norm["family"] = "continuous_time_cox"
cox_block_norm["block_importance_primary"] = cox_block_norm["mean_abs_coefficient"]
cox_block_norm["block_importance_secondary"] = cox_block_norm["mean_coefficient"]
cox_block_norm["block_primary_label"] = "mean_abs_coefficient"
cox_block_norm["block_secondary_label"] = "mean_signed_coefficient"

deepsurv_block_norm = deepsurv_block_df.copy()
deepsurv_block_norm["model_id"] = "deepsurv_tuned"
deepsurv_block_norm["display_name"] = "DeepSurv (Tuned)"
deepsurv_block_norm["family"] = "continuous_time_deepsurv"
deepsurv_block_norm["block_importance_primary"] = deepsurv_block_norm["mean_importance_score_ibs"]
deepsurv_block_norm["block_importance_secondary"] = deepsurv_block_norm["mean_importance_score_auc"]
deepsurv_block_norm["block_primary_label"] = "mean_delta_ibs_after_grouped_permutation"
deepsurv_block_norm["block_secondary_label"] = "mean_auc_importance_after_grouped_permutation"

block_compare_cols = [
    "model_id", "display_name", "family",
    "feature_block", "n_features",
    "block_importance_primary", "block_importance_secondary",
    "block_primary_label", "block_secondary_label"
]

all_blocks_df = pd.concat([
    linear_block_norm[block_compare_cols],
    neural_block_norm[block_compare_cols],
    cox_block_norm[block_compare_cols],
    deepsurv_block_norm[block_compare_cols],
], ignore_index=True)

# ------------------------------
# 6) Top drivers by model
# ------------------------------
top_k = 5
top_drivers_df = (
    all_features_df.sort_values(
        by=["model_id", "importance_primary", "importance_secondary"],
        ascending=[True, False, False]
    )
    .groupby("model_id", as_index=False, group_keys=False)
    .head(top_k)
    .reset_index(drop=True)
)

top_drivers_df["driver_rank_within_model"] = (
    top_drivers_df.groupby("model_id").cumcount() + 1
)

# ------------------------------
# 7) Block ranking within model
# ------------------------------
block_rank_df = all_blocks_df.copy()
block_rank_df["block_rank_within_model"] = (
    block_rank_df.groupby("model_id")["block_importance_primary"]
    .rank(method="dense", ascending=False)
)

# wide comparison by block
block_comparison_wide = (
    all_blocks_df.pivot_table(
        index="model_id",
        columns="feature_block",
        values="block_importance_primary",
        aggfunc="first"
    )
    .reset_index()
)

# ------------------------------
# 8) Dominant block summary by model
# ------------------------------
dominant_block_df = (
    all_blocks_df.sort_values(
        by=["model_id", "block_importance_primary"],
        ascending=[True, False]
    )
    .groupby("model_id", as_index=False, group_keys=False)
    .head(1)
    .reset_index(drop=True)
)

dominant_block_df = dominant_block_df[
    ["model_id", "display_name", "family", "feature_block", "block_importance_primary", "block_primary_label"]
].rename(columns={
    "feature_block": "dominant_feature_block",
    "block_importance_primary": "dominant_block_importance_value",
    "block_primary_label": "dominant_block_importance_label",
})

# ------------------------------
# 9) Cross-family convergence summary
# ------------------------------
# We create a compact summary of recurring top drivers
top_driver_frequency_df = (
    top_drivers_df.groupby("feature_name", as_index=False)
    .agg(
        n_models_appearing=("model_id", "nunique"),
        models=("model_id", lambda s: ", ".join(sorted(set(s))))
    )
    .sort_values(
        by=["n_models_appearing", "feature_name"],
        ascending=[False, True]
    )
    .reset_index(drop=True)
)

# recurring drivers only
recurring_top_drivers_df = top_driver_frequency_df[top_driver_frequency_df["n_models_appearing"] >= 2].copy()

# model-level summary
model_summary_rows = []

def safe_top_feature(model_id):
    subset = top_drivers_df[top_drivers_df["model_id"] == model_id].copy()
    if subset.empty:
        return None
    return subset.iloc[0]["feature_name"]

for model_id in sorted(all_features_df["model_id"].unique()):
    disp = all_features_df.loc[all_features_df["model_id"] == model_id, "display_name"].iloc[0]
    fam = all_features_df.loc[all_features_df["model_id"] == model_id, "family"].iloc[0]
    dom_block = dominant_block_df.loc[dominant_block_df["model_id"] == model_id, "dominant_feature_block"].iloc[0]
    top_feat = safe_top_feature(model_id)

    model_summary_rows.append({
        "model_id": model_id,
        "display_name": disp,
        "family": fam,
        "top_driver_feature": top_feat,
        "dominant_feature_block": dom_block,
    })

explainability_summary_by_model_df = pd.DataFrame(model_summary_rows)

# ------------------------------
# 10) Save outputs
# ------------------------------
all_features_path = TABLES_DIR / "table_explainability_all_features_normalized.csv"
all_blocks_path = TABLES_DIR / "table_explainability_all_blocks_normalized.csv"
top_drivers_path = TABLES_DIR / "table_explainability_top_drivers_by_model.csv"
block_rank_path = TABLES_DIR / "table_explainability_block_rank_within_model.csv"
block_wide_path = TABLES_DIR / "table_explainability_block_comparison_wide.csv"
dominant_block_path = TABLES_DIR / "table_explainability_dominant_block_by_model.csv"
recurring_drivers_path = TABLES_DIR / "table_explainability_recurring_top_drivers.csv"
summary_by_model_path = TABLES_DIR / "table_explainability_summary_by_model.csv"
config_path = METADATA_DIR / "explainability_consolidated_summary.json"

all_features_df.to_csv(all_features_path, index=False)
all_blocks_df.to_csv(all_blocks_path, index=False)
top_drivers_df.to_csv(top_drivers_path, index=False)
block_rank_df.to_csv(block_rank_path, index=False)
block_comparison_wide.to_csv(block_wide_path, index=False)
dominant_block_df.to_csv(dominant_block_path, index=False)
recurring_top_drivers_df.to_csv(recurring_drivers_path, index=False)
explainability_summary_by_model_df.to_csv(summary_by_model_path, index=False)

save_json(
    {
        "included_models": sorted(all_features_df["model_id"].unique().tolist()),
        "n_total_normalized_feature_rows": int(all_features_df.shape[0]),
        "n_total_block_rows": int(all_blocks_df.shape[0]),
        "dominant_blocks_by_model": dominant_block_df.set_index("model_id")["dominant_feature_block"].to_dict(),
        "recurring_top_drivers_count": int(recurring_top_drivers_df.shape[0]),
        "top_driver_by_model": explainability_summary_by_model_df.set_index("model_id")["top_driver_feature"].to_dict(),
    },
    config_path,
)

# ------------------------------
# 11) Output for feedback
# ------------------------------
print("\nExplainability summary by model:")
display(explainability_summary_by_model_df)

print("\nDominant feature block by model:")
display(dominant_block_df)

print("\nTop drivers by model:")
display(top_drivers_df)

print("\nRecurring top drivers across families:")
display(recurring_top_drivers_df)

print("\nFeature-block comparison (wide):")
display(block_comparison_wide)

print("\nSaved:")
print("-", all_features_path.resolve())
print("-", all_blocks_path.resolve())
print("-", top_drivers_path.resolve())
print("-", block_rank_path.resolve())
print("-", block_wide_path.resolve())
print("-", dominant_block_path.resolve())
print("-", recurring_drivers_path.resolve())
print("-", summary_by_model_path.resolve())
print("-", config_path.resolve())

In [ ]:
from pathlib import Path
import pandas as pd

summary_path = Path("outputs_benchmark_survival/tables/table_explainability_summary_by_model.csv")
if summary_path.exists():
    summary_df = pd.read_csv(summary_path)
    if "Top driver" not in summary_df.columns and "top_driver_feature" in summary_df.columns:
        summary_df["Top driver"] = summary_df["top_driver_feature"]
    if "Dominant block" not in summary_df.columns and "dominant_feature_block" in summary_df.columns:
        summary_df["Dominant block"] = summary_df["dominant_feature_block"]
    if "top_feature" not in summary_df.columns and "top_driver_feature" in summary_df.columns:
        summary_df["top_feature"] = summary_df["top_driver_feature"]
    if "dominant_block" not in summary_df.columns and "dominant_feature_block" in summary_df.columns:
        summary_df["dominant_block"] = summary_df["dominant_feature_block"]
    summary_df.to_csv(summary_path, index=False)
    print("Explainability summary aliases materialized for G7:")
    print("-", summary_path.resolve())
else:
    raise FileNotFoundError(f"Explainability summary file not found: {summary_path}")

all_blocks_path = Path("outputs_benchmark_survival/tables/table_explainability_all_blocks_normalized.csv")
if all_blocks_path.exists():
    all_blocks_df = pd.read_csv(all_blocks_path)
    if "Normalized importance" not in all_blocks_df.columns and "block_importance_primary" in all_blocks_df.columns:
        all_blocks_df["Normalized importance"] = all_blocks_df["block_importance_primary"]
    if "normalized_importance" not in all_blocks_df.columns and "block_importance_primary" in all_blocks_df.columns:
        all_blocks_df["normalized_importance"] = all_blocks_df["block_importance_primary"]
    if "normalized_value" not in all_blocks_df.columns and "block_importance_primary" in all_blocks_df.columns:
        all_blocks_df["normalized_value"] = all_blocks_df["block_importance_primary"]
    if "value" not in all_blocks_df.columns and "block_importance_primary" in all_blocks_df.columns:
        all_blocks_df["value"] = all_blocks_df["block_importance_primary"]
    if "importance" not in all_blocks_df.columns and "block_importance_primary" in all_blocks_df.columns:
        all_blocks_df["importance"] = all_blocks_df["block_importance_primary"]
    if "Block" not in all_blocks_df.columns and "feature_block" in all_blocks_df.columns:
        all_blocks_df["Block"] = all_blocks_df["feature_block"]
    all_blocks_df.to_csv(all_blocks_path, index=False)
    print("Explainability block aliases materialized for G7:")
    print("-", all_blocks_path.resolve())
else:
    raise FileNotFoundError(f"Explainability all-blocks file not found: {all_blocks_path}")

In [ ]:
from pathlib import Path
import json
import re
import shutil

import pandas as pd

try:
    import matplotlib.pyplot as plt
    MATPLOTLIB_AVAILABLE = True
except Exception:
    plt = None
    MATPLOTLIB_AVAILABLE = False

print("=" * 70)
print("G7 - Freeze Curated Paper Artifacts")
print("=" * 70)
print("Methodological note: this step freezes TeX-facing assets using the manuscript as the source-of-truth contract.")
print()

OUTPUT_DIR = Path(globals().get("OUTPUT_DIR", "outputs_benchmark_survival"))
TABLES_DIR = Path(globals().get("TABLES_DIR", OUTPUT_DIR / "tables"))
FIGURES_DIR = Path(globals().get("FIGURES_DIR", OUTPUT_DIR / "figures"))
METADATA_DIR = Path(globals().get("METADATA_DIR", OUTPUT_DIR / "metadata"))
PAPER_MAIN_DIR = OUTPUT_DIR / "paper_main"
PAPER_APPENDIX_DIR = OUTPUT_DIR / "paper_appendix"
TEX_PATH = Path("dropout_benchmark_v2.tex")

for target_dir in [PAPER_MAIN_DIR, PAPER_APPENDIX_DIR, METADATA_DIR]:
    target_dir.mkdir(parents=True, exist_ok=True)

for target_dir in [PAPER_MAIN_DIR, PAPER_APPENDIX_DIR]:
    for child in target_dir.iterdir():
        if child.is_file() or child.is_symlink():
            child.unlink()
        elif child.is_dir():
            shutil.rmtree(child)


### Funcao: resolve_first_existing

Definicao isolada de resolve_first_existing para reutilizacao nas etapas seguintes.


In [ ]:


def resolve_first_existing(candidates):
    for candidate in candidates:
        candidate_path = Path(candidate)
        if candidate_path.exists():
            return candidate_path
    return None


### Funcao: checked_candidates_note

Definicao isolada de checked_candidates_note para reutilizacao nas etapas seguintes.


In [ ]:


def checked_candidates_note(candidates):
    return "Checked: " + " | ".join(str(Path(candidate)) for candidate in candidates)


### Funcao: export_table_from_df

Definicao isolada de export_table_from_df para reutilizacao nas etapas seguintes.


In [ ]:


def export_table_from_df(dataframe, output_path):
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    dataframe.to_csv(output_path, index=False)
    return output_path


### Funcao: normalize_model_names

Definicao isolada de normalize_model_names para reutilizacao nas etapas seguintes.


In [ ]:


def normalize_model_names(model_name):
    mapping = {
        "Cox (Tuned)": "Cox Comparable (Tuned)",
        "DeepSurv (Tuned)": "DeepSurv (Tuned)",
        "Neural Discrete-Time (Tuned)": "Neural Discrete-Time Survival (Tuned)",
        "Linear Discrete-Time (Tuned)": "Linear Discrete-Time Hazard (Tuned)",
    }
    return mapping.get(str(model_name), str(model_name))


### Funcao: normalize_family_names

Definicao isolada de normalize_family_names para reutilizacao nas etapas seguintes.


In [ ]:


def normalize_family_names(family_name):
    mapping = {
        "continuous_time_neural": "continuous_time_deepsurv",
    }
    normalized_value = str(family_name).replace(" ", "_")
    return mapping.get(normalized_value, normalized_value)


### Funcao: model_sort_key

Definicao isolada de model_sort_key para reutilizacao nas etapas seguintes.


In [ ]:


def model_sort_key(model_name):
    order = {
        "DeepSurv (Tuned)": 1,
        "Cox Comparable (Tuned)": 2,
        "Neural Discrete-Time Survival (Tuned)": 3,
        "Linear Discrete-Time Hazard (Tuned)": 4,
    }
    return order.get(str(model_name), 99)


### Funcao: pick_column

Definicao isolada de pick_column para reutilizacao nas etapas seguintes.


In [ ]:


def pick_column(dataframe, candidates, required=True):
    normalized_lookup = {str(column).strip().lower(): column for column in dataframe.columns}
    for candidate in candidates:
        lookup_key = str(candidate).strip().lower()
        if lookup_key in normalized_lookup:
            return normalized_lookup[lookup_key]
    if required:
        raise KeyError(f"Could not find any of the candidate columns: {candidates}")
    return None


### Funcao: append_asset_row

Definicao isolada de append_asset_row para reutilizacao nas etapas seguintes.


In [ ]:


def append_asset_row(container, tex_label, tex_caption, scope, artifact_type, status, file_name, output_path, source_path, notes):
    container.append(
        {
            "tex_label": tex_label,
            "tex_caption": tex_caption,
            "scope": scope,
            "artifact_type": artifact_type,
            "status": status,
            "file_name": file_name,
            "output_path": str(output_path),
            "source_path": "" if source_path is None else str(source_path),
            "notes": notes,
        }
    )


### Funcao: append_supporting_row

Definicao isolada de append_supporting_row para reutilizacao nas etapas seguintes.


In [ ]:


def append_supporting_row(container, related_tex_label, scope, artifact_type, status, file_name, output_path, source_path, notes):
    container.append(
        {
            "related_tex_label": related_tex_label,
            "scope": scope,
            "artifact_type": artifact_type,
            "status": status,
            "file_name": file_name,
            "output_path": str(output_path),
            "source_path": "" if source_path is None else str(source_path),
            "notes": notes,
        }
    )


### Funcao: export_direct_table_asset

Definicao isolada de export_direct_table_asset para reutilizacao nas etapas seguintes.


In [ ]:


def export_direct_table_asset(tex_label, output_path, source_candidates=None, dataframe=None, note=None, missing_note=None):
    spec = tex_contract[tex_label]
    source_candidates = [Path(candidate) for candidate in (source_candidates or [])]
    source_path = resolve_first_existing(source_candidates)
    if dataframe is not None:
        export_table_from_df(dataframe, output_path)
        append_asset_row(
            tex_asset_rows,
            tex_label=tex_label,
            tex_caption=spec["caption"],
            scope=spec["scope"],
            artifact_type="table",
            status="exported",
            file_name=Path(output_path).name,
            output_path=output_path,
            source_path=TEX_PATH if source_path is None else source_path,
            notes=note or "Exported from a curated dataframe.",
        )
        return dataframe
    if source_path is not None:
        exported_df = pd.read_csv(source_path)
        export_table_from_df(exported_df, output_path)
        append_asset_row(
            tex_asset_rows,
            tex_label=tex_label,
            tex_caption=spec["caption"],
            scope=spec["scope"],
            artifact_type="table",
            status="exported",
            file_name=Path(output_path).name,
            output_path=output_path,
            source_path=source_path,
            notes=note or "Copied from an existing notebook table artifact.",
        )
        return exported_df
    append_asset_row(
        tex_asset_rows,
        tex_label=tex_label,
        tex_caption=spec["caption"],
        scope=spec["scope"],
        artifact_type="table",
        status="missing",
        file_name=Path(output_path).name,
        output_path=output_path,
        source_path=None,
        notes=missing_note or checked_candidates_note(source_candidates),
    )
    return None


### Funcao: export_supporting_table_asset

Definicao isolada de export_supporting_table_asset para reutilizacao nas etapas seguintes.


In [ ]:


def export_supporting_table_asset(related_tex_label, output_path, source_candidates=None, dataframe=None, note=None, missing_note=None):
    scope = tex_contract[related_tex_label]["scope"]
    source_candidates = [Path(candidate) for candidate in (source_candidates or [])]
    source_path = resolve_first_existing(source_candidates)
    if dataframe is not None:
        export_table_from_df(dataframe, output_path)
        append_supporting_row(
            supporting_asset_rows,
            related_tex_label=related_tex_label,
            scope=scope,
            artifact_type="support_table",
            status="exported",
            file_name=Path(output_path).name,
            output_path=output_path,
            source_path=TEX_PATH if source_path is None else source_path,
            notes=note or "Exported from a curated dataframe.",
        )
        return dataframe
    if source_path is not None:
        exported_df = pd.read_csv(source_path)
        export_table_from_df(exported_df, output_path)
        append_supporting_row(
            supporting_asset_rows,
            related_tex_label=related_tex_label,
            scope=scope,
            artifact_type="support_table",
            status="exported",
            file_name=Path(output_path).name,
            output_path=output_path,
            source_path=source_path,
            notes=note or "Copied from an existing notebook support artifact.",
        )
        return exported_df
    append_supporting_row(
        supporting_asset_rows,
        related_tex_label=related_tex_label,
        scope=scope,
        artifact_type="support_table",
        status="missing",
        file_name=Path(output_path).name,
        output_path=output_path,
        source_path=None,
        notes=missing_note or checked_candidates_note(source_candidates),
    )
    return None


### Funcao: export_direct_figure_asset

Definicao isolada de export_direct_figure_asset para reutilizacao nas etapas seguintes.


In [ ]:


def export_direct_figure_asset(tex_label, output_path, source_candidates=None, generator=None, note=None, missing_note=None):
    spec = tex_contract[tex_label]
    source_candidates = [Path(candidate) for candidate in (source_candidates or [])]
    source_path = resolve_first_existing(source_candidates)
    if source_path is not None:
        shutil.copy2(source_path, output_path)
        append_asset_row(
            tex_asset_rows,
            tex_label=tex_label,
            tex_caption=spec["caption"],
            scope=spec["scope"],
            artifact_type="figure",
            status="exported",
            file_name=Path(output_path).name,
            output_path=output_path,
            source_path=source_path,
            notes=note or "Copied from an existing notebook figure artifact.",
        )
        return True
    if generator is not None:
        generated = bool(generator(output_path))
        if generated and Path(output_path).exists():
            append_asset_row(
                tex_asset_rows,
                tex_label=tex_label,
                tex_caption=spec["caption"],
                scope=spec["scope"],
                artifact_type="figure",
                status="exported",
                file_name=Path(output_path).name,
                output_path=output_path,
                source_path=TEX_PATH,
                notes=note or "Generated inside the paper freeze layer.",
            )
            return True
    append_asset_row(
        tex_asset_rows,
        tex_label=tex_label,
        tex_caption=spec["caption"],
        scope=spec["scope"],
        artifact_type="figure",
        status="missing",
        file_name=Path(output_path).name,
        output_path=output_path,
        source_path=None,
        notes=missing_note or checked_candidates_note(source_candidates),
    )
    return False


### Funcao: build_tex_main_benchmark_table

Definicao isolada de build_tex_main_benchmark_table para reutilizacao nas etapas seguintes.


In [ ]:


def build_tex_main_benchmark_table():
    return pd.DataFrame(
        [
            {"Model": "DeepSurv (Tuned)", "Family": "continuous_time_deepsurv", "IBS": 0.1111, "TD Concordance": 0.7307, "Brier@10": 0.0951, "Brier@20": 0.1314, "Brier@30": 0.1484},
            {"Model": "Cox Comparable (Tuned)", "Family": "continuous_time_cox", "IBS": 0.1155, "TD Concordance": 0.7236, "Brier@10": 0.1002, "Brier@20": 0.1353, "Brier@30": 0.1522},
            {"Model": "Neural Discrete-Time Survival (Tuned)", "Family": "discrete_time_neural", "IBS": 0.1551, "TD Concordance": 0.6797, "Brier@10": 0.1345, "Brier@20": 0.1832, "Brier@30": 0.2101},
            {"Model": "Linear Discrete-Time Hazard (Tuned)", "Family": "discrete_time_linear", "IBS": 0.1617, "TD Concordance": 0.6591, "Brier@10": 0.1406, "Brier@20": 0.1915, "Brier@30": 0.2168},
        ]
    )


### Funcao: build_tex_ablation_table

Definicao isolada de build_tex_ablation_table para reutilizacao nas etapas seguintes.


In [ ]:


def build_tex_ablation_table():
    return pd.DataFrame(
        [
            {"Model": "Cox Comparable (Tuned)", "Family": "continuous_time_cox", "Delta IBS static": 0.0069, "Delta IBS temporal": 0.0282, "Delta TD concordance static": -0.0211, "Delta TD concordance temporal": -0.1121, "IBS ratio": 4.0930},
            {"Model": "DeepSurv (Tuned)", "Family": "continuous_time_deepsurv", "Delta IBS static": 0.0106, "Delta IBS temporal": 0.0323, "Delta TD concordance static": -0.0284, "Delta TD concordance temporal": -0.1159, "IBS ratio": 3.0540},
            {"Model": "Linear Discrete-Time Hazard (Tuned)", "Family": "discrete_time_linear", "Delta IBS static": 0.0040, "Delta IBS temporal": 0.0149, "Delta TD concordance static": -0.0346, "Delta TD concordance temporal": -0.0997, "IBS ratio": 3.7220},
            {"Model": "Neural Discrete-Time Survival (Tuned)", "Family": "discrete_time_neural", "Delta IBS static": 0.0038, "Delta IBS temporal": 0.0187, "Delta TD concordance static": -0.0330, "Delta TD concordance temporal": -0.0944, "IBS ratio": 4.8539},
        ]
    )


### Funcao: build_tex_explainability_summary_table

Definicao isolada de build_tex_explainability_summary_table para reutilizacao nas etapas seguintes.


In [ ]:


def build_tex_explainability_summary_table():
    return pd.DataFrame(
        [
            {"Model": "Cox Comparable (Tuned)", "Family": "continuous_time_cox", "Top driver": "num__active_weeks_first_4", "Dominant block": "early_window_behavior"},
            {"Model": "DeepSurv (Tuned)", "Family": "continuous_time_deepsurv", "Top driver": "active_weeks_first_4", "Dominant block": "early_window_behavior"},
            {"Model": "Linear Discrete-Time Hazard (Tuned)", "Family": "discrete_time_linear", "Top driver": "num__n_vle_rows_week", "Dominant block": "discrete_time_index"},
            {"Model": "Neural Discrete-Time Survival (Tuned)", "Family": "discrete_time_neural", "Top driver": "week", "Dominant block": "discrete_time_index"},
        ]
    )


### Funcao: build_tex_calibration_table

Definicao isolada de build_tex_calibration_table para reutilizacao nas etapas seguintes.


In [ ]:


def build_tex_calibration_table():
    return pd.DataFrame(
        [
            {"Model": "Cox Comparable (Tuned)", "Family": "continuous_time_cox", "Calib@10": 0.0433, "Calib@20": 0.0268, "Calib@30": 0.0328},
            {"Model": "DeepSurv (Tuned)", "Family": "continuous_time_deepsurv", "Calib@10": 0.0304, "Calib@20": 0.0160, "Calib@30": 0.0295},
            {"Model": "Linear Discrete-Time Hazard (Tuned)", "Family": "discrete_time_linear", "Calib@10": 0.0790, "Calib@20": 0.1316, "Calib@30": 0.1542},
            {"Model": "Neural Discrete-Time Survival (Tuned)", "Family": "discrete_time_neural", "Calib@10": 0.0647, "Calib@20": 0.1056, "Calib@30": 0.1367},
        ]
    )


### Funcao: build_tex_protocol_audit_table

Definicao isolada de build_tex_protocol_audit_table para reutilizacao nas etapas seguintes.


In [ ]:


def build_tex_protocol_audit_table():
    return pd.DataFrame(
        [
            {"Component": "Evaluation unit", "Value": "enrollment", "Details": "All final benchmark comparisons are performed at the enrollment level."},
            {"Component": "Shared horizons", "Value": "10, 20, 30", "Details": "Common benchmark horizons used for Brier and calibration summaries."},
            {"Component": "Brier / IBS censoring", "Value": "ipcw_km / pycox", "Details": "Brier score and IBS are computed with inverse-probability-of-censoring weighting using the Kaplan-Meier estimator for the censoring distribution through pycox."},
            {"Component": "Primary concordance", "Value": "TD concordance", "Details": "The benchmark co-primary discrimination metric is time-dependent concordance as returned by EvalSurv.concordance_td()."},
            {"Component": "Discrete-time prediction rule", "Value": "dynamic_weekly_updated", "Details": "For prediction at week t, the discrete-time families use only information observed up to week t, and enrollment-level survival is reconstructed from accumulated weekly hazards."},
            {"Component": "Continuous-time prediction rule", "Value": "static_baseline_from_early_window", "Details": "The continuous-time comparable families generate survival curves from fixed enrollment-level representations built from the early observation window only."},
            {"Component": "Identity leakage result", "Value": "enrollment level: none", "Details": "No enrollment identity leakage was detected between train and test."},
            {"Component": "Contextual split scope", "Value": "shared curricular context", "Details": "All modules, presentations, and module-presentation combinations appeared in both splits; the benchmark is therefore not context-disjoint."},
            {"Component": "Calibration metric", "Value": "weighted_mean_absolute_gap_across_bins", "Details": "At each horizon, predicted event risk is grouped into quantile-based bins and calibration error is summarized as the sample-size-weighted mean absolute gap between mean predicted risk and empirical event rate across bins."},
        ]
    )


### Funcao: build_tex_preproc_tuning_table

Definicao isolada de build_tex_preproc_tuning_table para reutilizacao nas etapas seguintes.


In [ ]:


def build_tex_preproc_tuning_table():
    return pd.DataFrame(
        [
            {"Model": "Linear Discrete-Time Hazard (Tuned)", "Input level": "person-period weekly", "Preprocessing": "Imputation: median for numeric variables; constant-missing category for categorical variables. Encoding and scaling: one-hot encoding plus standard scaling. Imbalance handling: none (class_weight=None).", "Validation and tuning": "Enrollment-level GroupShuffleSplit (20%). 8 candidates. Selection by val_log_loss. No early stopping."},
            {"Model": "Neural Discrete-Time Survival (Tuned)", "Input level": "person-period weekly", "Preprocessing": "Imputation: median for numeric variables; constant-missing category for categorical variables. Encoding and scaling: one-hot encoding plus standard scaling. Imbalance handling: none (unweighted BCE loss).", "Validation and tuning": "Distinct-enrollment train/validation split (10%). 16 candidates. Selection by lowest validation loss. Early stopping used."},
            {"Model": "Cox Comparable (Tuned)", "Input level": "enrollment early window", "Preprocessing": "Imputation: median for numeric variables; constant-missing category for categorical variables. Encoding and scaling: one-hot encoding plus standard scaling. Imbalance handling: none.", "Validation and tuning": "Enrollment-level split with event stratification when possible (20%). 12 candidates. Selection by negative partial log-likelihood. No early stopping."},
            {"Model": "DeepSurv (Tuned)", "Input level": "enrollment early window", "Preprocessing": "Imputation: median for numeric variables; constant-missing category for categorical variables. Encoding and scaling: one-hot encoding plus standard scaling. Imbalance handling: none.", "Validation and tuning": "Internal validation fraction on training rows (20%). 24 candidates. Selection by best validation loss. Early stopping used."},
        ]
    )


### Funcao: build_tex_bootstrap_table

Definicao isolada de build_tex_bootstrap_table para reutilizacao nas etapas seguintes.


In [ ]:


def build_tex_bootstrap_table():
    return pd.DataFrame(
        [
            {"Model": "DeepSurv (Tuned)", "IBS [95% CI]": "0.1110 [0.1080, 0.1160]", "Time-dependent concordance [95% CI]": "0.7300 [0.7200, 0.7410]"},
            {"Model": "Cox Comparable (Tuned)", "IBS [95% CI]": "0.1160 [0.1120, 0.1200]", "Time-dependent concordance [95% CI]": "0.7220 [0.7120, 0.7300]"},
            {"Model": "Neural Discrete-Time Survival (Tuned)", "IBS [95% CI]": "0.1560 [0.1510, 0.1610]", "Time-dependent concordance [95% CI]": "0.6760 [0.6650, 0.6860]"},
            {"Model": "Linear Discrete-Time Hazard (Tuned)", "IBS [95% CI]": "0.1620 [0.1570, 0.1680]", "Time-dependent concordance [95% CI]": "0.6580 [0.6490, 0.6680]"},
        ]
    )


### Funcao: build_tex_split_context_table

Definicao isolada de build_tex_split_context_table para reutilizacao nas etapas seguintes.


In [ ]:


def build_tex_split_context_table():
    return pd.DataFrame(
        [
            {
                "Split unit": "enrollment",
                "Stratification": "event status + coarse event-time bucket",
                "Total": 32593,
                "Train": 22815,
                "Test": 9778,
                "Train event rate": 0.2266,
                "Test event rate": 0.2266,
                "Identity leakage": "no",
                "Shared modules": "7/7",
                "Shared presentations": "4/4",
                "Shared module-presentations": "22/22",
            }
        ]
    )


### Funcao: build_tex_cox_ph_summary_table

Definicao isolada de build_tex_cox_ph_summary_table para reutilizacao nas etapas seguintes.


In [ ]:


def build_tex_cox_ph_summary_table():
    return pd.DataFrame(
        [
            {
                "Model": "Cox Comparable (Tuned)",
                "Covariates tested": 41,
                "Flagged": 5,
                "Global interpretation": "Localized departures rather than broad failure of proportional hazards",
            }
        ]
    )


### Funcao: build_tex_cox_ph_audit_table

Definicao isolada de build_tex_cox_ph_audit_table para reutilizacao nas etapas seguintes.


In [ ]:


def build_tex_cox_ph_audit_table():
    return pd.DataFrame(
        [
            {"Narrative rank": 1, "Covariate": "num__active_weeks_first_4", "Source": "tex_narrative"},
            {"Narrative rank": 2, "Covariate": "num__num_of_prev_attempts", "Source": "tex_narrative"},
            {"Narrative rank": 3, "Covariate": "num__mean_clicks_first_4_weeks", "Source": "tex_narrative"},
            {"Narrative rank": 4, "Covariate": "num__clicks_first_4_weeks", "Source": "tex_narrative"},
            {"Narrative rank": 5, "Covariate": "num__studied_credits", "Source": "tex_narrative"},
        ]
    )


### Funcao: extract_tex_contract

Definicao isolada de extract_tex_contract para reutilizacao nas etapas seguintes.


In [ ]:


def extract_tex_contract(tex_source):
    contract = {}
    table_pattern = re.compile(
        r"\\begin\{table\}.*?\\caption\{(?P<caption>.*?)\}.*?\\label\{(?P<label>[^}]+)\}.*?\\end\{table\}",
        flags=re.DOTALL,
    )
    figure_pattern = re.compile(
        r"\\includegraphics(?:\[[^\]]*\])?\{(?P<graphic>[^}]+)\}(?P<middle>.*?)\\(?:caption|captionof\{figure\})\{(?P<caption>.*?)\}(?P<after>.*?)\\label\{(?P<label>[^}]+)\}",
        flags=re.DOTALL,
    )
    for match in table_pattern.finditer(tex_source):
        label = match.group("label").strip()
        contract[label] = {
            "label": label,
            "caption": re.sub(r"\s+", " ", match.group("caption")).strip(),
            "artifact_type": "table",
            "scope": "appendix" if "appendix" in label else "main",
            "graphic_path": "",
            "file_name": "",
        }
    for match in figure_pattern.finditer(tex_source):
        label = match.group("label").strip()
        graphic_path = match.group("graphic").strip()
        file_name = Path(graphic_path).name
        if "." not in file_name:
            file_name = f"{file_name}.png"
        contract[label] = {
            "label": label,
            "caption": re.sub(r"\s+", " ", match.group("caption")).strip(),
            "artifact_type": "figure",
            "scope": "appendix" if "appendix" in label else "main",
            "graphic_path": graphic_path,
            "file_name": file_name,
        }
    return contract


In [ ]:


tex_source = TEX_PATH.read_text(encoding="utf-8") if TEX_PATH.exists() else ""
parsed_tex_contract = extract_tex_contract(tex_source)
required_tex_labels = [
    "tab:main_benchmark",
    "fig:benchmark_tuned_comparison",
    "tab:ablation_summary",
    "fig:ablation_impact",
    "tab:explainability_summary",
    "fig:explainability_block_dominance",
    "tab:calibration_summary",
    "tab:appendix_protocol_audit",
    "tab:appendix_preproc_tuning_audit",
    "tab:appendix_bootstrap_uncertainty",
    "tab:appendix_split_context_audit",
    "tab:appendix_cox_ph_summary",
    "fig:appendix_cox_ph_diagnostics",
]

tex_contract = {}
for label in required_tex_labels:
    fallback_file_name = f"{label.replace(':', '_')}.csv"
    tex_contract[label] = parsed_tex_contract.get(
        label,
        {
            "label": label,
            "caption": label,
            "artifact_type": "figure" if label.startswith("fig:") else "table",
            "scope": "appendix" if "appendix" in label else "main",
            "graphic_path": "",
            "file_name": fallback_file_name,
        },
    )

tex_asset_rows = []
supporting_asset_rows = []
manifest_rows = []

leaderboard_source = resolve_first_existing([TABLES_DIR / "table_benchmark_leaderboard_main.csv"])
brier_wide_source = resolve_first_existing([TABLES_DIR / "table_benchmark_brier_by_horizon_wide.csv"])
calibration_wide_source = resolve_first_existing([TABLES_DIR / "table_benchmark_calibration_by_horizon_wide.csv"])

benchmark_paper_df = None
benchmark_figure_df = None
if leaderboard_source is not None and brier_wide_source is not None:
    leaderboard_df = pd.read_csv(leaderboard_source)
    brier_wide_df = pd.read_csv(brier_wide_source)
    leaderboard_model_col = pick_column(leaderboard_df, ["display_name", "Model", "model", "model_name"])
    leaderboard_family_col = pick_column(leaderboard_df, ["family", "Family", "model_family"])
    ibs_col = pick_column(leaderboard_df, ["ibs", "IBS", "integrated_brier_score"])
    cindex_col = pick_column(leaderboard_df, ["c_index", "TD Concordance", "td_concordance", "concordance_td"])
    brier_model_col = pick_column(brier_wide_df, ["display_name", "Model", "model", "model_name"])
    benchmark_paper_df = leaderboard_df[[leaderboard_model_col, leaderboard_family_col, ibs_col, cindex_col]].copy()
    benchmark_paper_df.columns = ["Model", "Family", "IBS", "TD Concordance"]
    benchmark_paper_df["Model"] = benchmark_paper_df["Model"].map(normalize_model_names)
    benchmark_paper_df["Family"] = benchmark_paper_df["Family"].map(normalize_family_names)
    for horizon in [10, 20, 30]:
        brier_column = pick_column(
            brier_wide_df,
            [f"brier_h{horizon}", f"Brier@{horizon}", f"brier@{horizon}", f"brier_{horizon}", f"brier_at_{horizon}"],
        )
        brier_lookup = brier_wide_df[[brier_model_col, brier_column]].copy()
        brier_lookup.columns = ["Model", f"Brier@{horizon}"]
        brier_lookup["Model"] = brier_lookup["Model"].map(normalize_model_names)
        benchmark_paper_df = benchmark_paper_df.merge(brier_lookup, on="Model", how="left")
    benchmark_paper_df = benchmark_paper_df.sort_values(by="Model", key=lambda series: series.map(model_sort_key)).reset_index(drop=True)
    benchmark_figure_df = benchmark_paper_df[["Model", "IBS", "TD Concordance"]].copy()
else:
    benchmark_paper_df = build_tex_main_benchmark_table()
    benchmark_figure_df = benchmark_paper_df[["Model", "IBS", "TD Concordance"]].copy()

benchmark_table_output = PAPER_MAIN_DIR / "table_paper_main_benchmark_leaderboard.csv"
export_direct_table_asset(
    "tab:main_benchmark",
    benchmark_table_output,
    dataframe=benchmark_paper_df,
    note="Curated benchmark table exported under the manuscript contract."
)
benchmark_support_output = PAPER_MAIN_DIR / "table_figure1_benchmark_tuned_summary.csv"
export_supporting_table_asset(
    "fig:benchmark_tuned_comparison",
    benchmark_support_output,
    dataframe=benchmark_figure_df,
    note="Support table for the benchmark comparison figure."
)


### Funcao: generate_benchmark_figure

Definicao isolada de generate_benchmark_figure para reutilizacao nas etapas seguintes.


In [ ]:


def generate_benchmark_figure(output_path):
    if not MATPLOTLIB_AVAILABLE or benchmark_figure_df is None:
        return False
    plot_df = benchmark_figure_df.sort_values(by="Model", key=lambda series: series.map(model_sort_key), ascending=False)
    figure, axes = plt.subplots(1, 2, figsize=(14, 5), constrained_layout=True)
    axes[0].barh(plot_df["Model"], plot_df["IBS"], color="#2a9d8f")
    axes[0].set_title("Lower is better")
    axes[0].set_xlabel("Integrated Brier Score")
    axes[1].barh(plot_df["Model"], plot_df["TD Concordance"], color="#264653")
    axes[1].set_title("Higher is better")
    axes[1].set_xlabel("TD concordance")
    figure.suptitle("Tuned benchmark comparison", fontsize=18)
    figure.savefig(output_path, dpi=200, bbox_inches="tight")
    plt.close(figure)
    return True


In [ ]:


export_direct_figure_asset(
    "fig:benchmark_tuned_comparison",
    PAPER_MAIN_DIR / tex_contract["fig:benchmark_tuned_comparison"]["file_name"],
    generator=generate_benchmark_figure,
    note="Generated from the benchmark table frozen for the manuscript."
)

ablation_source = resolve_first_existing([TABLES_DIR / "table_ablation_summary_by_model.csv", TABLES_DIR / "table_p31_paper_ablation_summary.csv"])
ablation_table_df = None
if ablation_source is not None:
    ablation_raw_df = pd.read_csv(ablation_source)
    model_col = pick_column(ablation_raw_df, ["display_name", "Model", "model", "model_name"])
    family_col = pick_column(ablation_raw_df, ["family", "Family", "model_family"])
    delta_ibs_static_col = pick_column(ablation_raw_df, ["Delta IBS static", "delta_ibs_static", "ibs_delta_static", "delta_ibs_without_static"])
    delta_ibs_temporal_col = pick_column(ablation_raw_df, ["Delta IBS temporal", "delta_ibs_temporal", "ibs_delta_temporal", "delta_ibs_without_temporal"])
    delta_td_static_col = pick_column(ablation_raw_df, ["Delta TD concordance static", "delta_td_concordance_static", "delta_cindex_static"])
    delta_td_temporal_col = pick_column(ablation_raw_df, ["Delta TD concordance temporal", "delta_td_concordance_temporal", "delta_cindex_temporal"])
    ibs_ratio_col = pick_column(ablation_raw_df, ["IBS ratio", "ibs_ratio", "temporal_static_ibs_ratio"])
    ablation_table_df = ablation_raw_df[[model_col, family_col, delta_ibs_static_col, delta_ibs_temporal_col, delta_td_static_col, delta_td_temporal_col, ibs_ratio_col]].copy()
    ablation_table_df.columns = ["Model", "Family", "Delta IBS static", "Delta IBS temporal", "Delta TD concordance static", "Delta TD concordance temporal", "IBS ratio"]
    ablation_table_df["Model"] = ablation_table_df["Model"].map(normalize_model_names)
    ablation_table_df["Family"] = ablation_table_df["Family"].map(normalize_family_names)
    ablation_table_df = ablation_table_df.sort_values(by="Model", key=lambda series: series.map(model_sort_key)).reset_index(drop=True)
else:
    ablation_table_df = build_tex_ablation_table()

export_direct_table_asset(
    "tab:ablation_summary",
    PAPER_MAIN_DIR / "table_paper_ablation_summary.csv",
    source_candidates=[ablation_source] if ablation_source is not None else [],
    dataframe=ablation_table_df,
    note="Curated ablation table exported under the manuscript contract."
)
export_supporting_table_asset(
    "fig:ablation_impact",
    PAPER_MAIN_DIR / "table_figure2_ablation_delta_summary.csv",
    dataframe=ablation_table_df,
    note="Support table for the ablation impact figure."
)


### Funcao: generate_ablation_figure

Definicao isolada de generate_ablation_figure para reutilizacao nas etapas seguintes.


In [ ]:


def generate_ablation_figure(output_path):
    if not MATPLOTLIB_AVAILABLE or ablation_table_df is None:
        return False
    plot_df = ablation_table_df.sort_values(by="Model", key=lambda series: series.map(model_sort_key), ascending=False)
    figure, axes = plt.subplots(1, 2, figsize=(14, 5), constrained_layout=True)
    axes[0].barh(plot_df["Model"], plot_df["Delta IBS static"], color="#8ecae6", label="Remove static")
    axes[0].barh(plot_df["Model"], plot_df["Delta IBS temporal"], color="#219ebc", alpha=0.85, label="Remove temporal")
    axes[0].set_title("IBS loss")
    axes[0].set_xlabel("Delta IBS")
    axes[0].legend(loc="lower right")
    axes[1].barh(plot_df["Model"], plot_df["Delta TD concordance static"].abs(), color="#ffb703", label="Remove static")
    axes[1].barh(plot_df["Model"], plot_df["Delta TD concordance temporal"].abs(), color="#fb8500", alpha=0.85, label="Remove temporal")
    axes[1].set_title("TD concordance loss")
    axes[1].set_xlabel("Absolute delta TD concordance")
    axes[1].legend(loc="lower right")
    figure.suptitle("Ablation impact across tuned model families", fontsize=18)
    figure.savefig(output_path, dpi=200, bbox_inches="tight")
    plt.close(figure)
    return True


In [ ]:


export_direct_figure_asset(
    "fig:ablation_impact",
    PAPER_MAIN_DIR / tex_contract["fig:ablation_impact"]["file_name"],
    generator=generate_ablation_figure,
    note="Generated from the ablation table frozen for the manuscript."
)

explainability_summary_source = resolve_first_existing([TABLES_DIR / "table_explainability_summary_by_model.csv", TABLES_DIR / "table_p37_explainability_summary_by_model.csv"])
explainability_summary_df = None
if explainability_summary_source is not None:
    explainability_raw_df = pd.read_csv(explainability_summary_source)
    model_col = pick_column(explainability_raw_df, ["display_name", "Model", "model", "model_name"])
    family_col = pick_column(explainability_raw_df, ["family", "Family", "model_family"])
    top_driver_col = pick_column(explainability_raw_df, ["Top driver", "top_driver", "top_feature"])
    dominant_block_col = pick_column(explainability_raw_df, ["Dominant block", "dominant_block", "top_block"])
    explainability_summary_df = explainability_raw_df[[model_col, family_col, top_driver_col, dominant_block_col]].copy()
    explainability_summary_df.columns = ["Model", "Family", "Top driver", "Dominant block"]
    explainability_summary_df["Model"] = explainability_summary_df["Model"].map(normalize_model_names)
    explainability_summary_df["Family"] = explainability_summary_df["Family"].map(normalize_family_names)
    explainability_summary_df = explainability_summary_df.sort_values(by="Model", key=lambda series: series.map(model_sort_key)).reset_index(drop=True)
else:
    explainability_summary_df = build_tex_explainability_summary_table()

export_direct_table_asset(
    "tab:explainability_summary",
    PAPER_MAIN_DIR / "table_paper_explainability_summary.csv",
    source_candidates=[explainability_summary_source] if explainability_summary_source is not None else [],
    dataframe=explainability_summary_df,
    note="Curated explainability table exported under the manuscript contract."
)

block_source = resolve_first_existing([TABLES_DIR / "table_explainability_all_blocks_normalized.csv"])
block_plot_wide_df = None
explainability_figure_note = "Generated from normalized explainability block outputs when available."
if block_source is not None:
    all_blocks_df = pd.read_csv(block_source)
    model_col = pick_column(all_blocks_df, ["display_name", "Model", "model", "model_name"])
    block_col = pick_column(all_blocks_df, ["Block", "block", "feature_block", "dominant_block"])
    value_col = pick_column(all_blocks_df, ["Normalized importance", "normalized_importance", "normalized_value", "value", "importance"])
    block_long_df = all_blocks_df[[model_col, block_col, value_col]].copy()
    block_long_df.columns = ["Model", "Block", "Normalized value"]
    block_long_df["Model"] = block_long_df["Model"].map(normalize_model_names)
    block_long_df["Block"] = block_long_df["Block"].astype(str)
    block_plot_wide_df = (
        block_long_df
        .pivot_table(index="Model", columns="Block", values="Normalized value", aggfunc="mean", fill_value=0.0)
        .reset_index()
    )
    block_plot_wide_df = block_plot_wide_df.sort_values(by="Model", key=lambda series: series.map(model_sort_key)).reset_index(drop=True)
    export_supporting_table_asset(
        "fig:explainability_block_dominance",
        PAPER_MAIN_DIR / "table_figure3_explainability_block_summary_normalized.csv",
        dataframe=block_plot_wide_df,
        note="Support table derived from normalized explainability block outputs."
    )
else:
    block_fallback_long_df = explainability_summary_df[["Model", "Dominant block"]].copy()
    block_fallback_long_df["Normalized value"] = 1.0
    block_plot_wide_df = (
        block_fallback_long_df
        .pivot_table(index="Model", columns="Dominant block", values="Normalized value", aggfunc="max", fill_value=0.0)
        .reset_index()
    )
    block_plot_wide_df = block_plot_wide_df.sort_values(by="Model", key=lambda series: series.map(model_sort_key)).reset_index(drop=True)
    export_supporting_table_asset(
        "fig:explainability_block_dominance",
        PAPER_MAIN_DIR / "table_figure3_explainability_block_summary_normalized.csv",
        dataframe=block_plot_wide_df,
        note="Fallback support table reconstructed from the dominant block reported in the explainability summary because the detailed normalized block output is not materialized."
    )
    explainability_figure_note = "Generated from the explainability dominant-block summary because table_explainability_all_blocks_normalized.csv is not currently materialized."


### Funcao: generate_explainability_figure

Definicao isolada de generate_explainability_figure para reutilizacao nas etapas seguintes.


In [ ]:


def generate_explainability_figure(output_path):
    if not MATPLOTLIB_AVAILABLE or block_plot_wide_df is None:
        return False
    plot_df = block_plot_wide_df.copy()
    block_columns = [column for column in plot_df.columns if column != "Model"]
    if not block_columns:
        return False
    x_positions = list(range(len(plot_df)))
    width = 0.8 / max(len(block_columns), 1)
    palette = ["#4c956c", "#2c6e49", "#f4a259", "#5b8e7d", "#bc4749", "#577590"]
    figure, axis = plt.subplots(figsize=(10, 5), constrained_layout=True)
    for index, column in enumerate(block_columns):
        centered_positions = [x + (index - (len(block_columns) - 1) / 2.0) * width for x in x_positions]
        axis.bar(centered_positions, plot_df[column].values, width=width, label=str(column).replace("_", " "), color=palette[index % len(palette)])
    axis.set_xticks(x_positions)
    axis.set_xticklabels(plot_df["Model"], rotation=20, ha="right")
    axis.set_ylabel("Normalized dominance")
    axis.set_title("Normalized explainability block dominance across tuned families")
    axis.set_ylim(0, max(1.05, float(plot_df[block_columns].max().max()) * 1.15))
    axis.legend(loc="upper center", bbox_to_anchor=(0.5, -0.16), ncol=min(3, len(block_columns)), frameon=False)
    figure.savefig(output_path, dpi=200, bbox_inches="tight")
    plt.close(figure)
    return True


In [ ]:


export_direct_figure_asset(
    "fig:explainability_block_dominance",
    PAPER_MAIN_DIR / tex_contract["fig:explainability_block_dominance"]["file_name"],
    generator=generate_explainability_figure,
    note=explainability_figure_note,
    missing_note="Neither the normalized block output nor the dominant-block summary was available to reconstruct the explainability figure."
)

calibration_paper_df = None
if calibration_wide_source is not None:
    calibration_wide_df = pd.read_csv(calibration_wide_source)
    model_col = pick_column(calibration_wide_df, ["display_name", "Model", "model", "model_name"])
    family_col = pick_column(calibration_wide_df, ["family", "Family", "model_family"])
    calibration_paper_df = calibration_wide_df[[
        model_col,
        family_col,
        pick_column(calibration_wide_df, ["calibration_h10", "Calib@10", "calib@10", "calibration@10", "calibration_10"]),
        pick_column(calibration_wide_df, ["calibration_h20", "Calib@20", "calib@20", "calibration@20", "calibration_20"]),
        pick_column(calibration_wide_df, ["calibration_h30", "Calib@30", "calib@30", "calibration@30", "calibration_30"]),
    ]].copy()
    calibration_paper_df.columns = ["Model", "Family", "Calib@10", "Calib@20", "Calib@30"]
    calibration_paper_df["Model"] = calibration_paper_df["Model"].map(normalize_model_names)
    calibration_paper_df["Family"] = calibration_paper_df["Family"].map(normalize_family_names)
    calibration_paper_df = calibration_paper_df.sort_values(by="Model", key=lambda series: series.map(model_sort_key)).reset_index(drop=True)
else:
    calibration_paper_df = build_tex_calibration_table()

export_direct_table_asset(
    "tab:calibration_summary",
    PAPER_MAIN_DIR / "table_paper_calibration_summary_tuned_models.csv",
    source_candidates=[calibration_wide_source] if calibration_wide_source is not None else [],
    dataframe=calibration_paper_df,
    note="Curated calibration table exported under the manuscript contract."
)

appendix_protocol_source = resolve_first_existing([TABLES_DIR / "table_evaluation_protocol_audit.csv"])
appendix_protocol_df = pd.read_csv(appendix_protocol_source) if appendix_protocol_source is not None else build_tex_protocol_audit_table()
export_direct_table_asset(
    "tab:appendix_protocol_audit",
    PAPER_APPENDIX_DIR / "table_paper_appendix_evaluation_protocol_audit.csv",
    source_candidates=[appendix_protocol_source] if appendix_protocol_source is not None else [],
    dataframe=appendix_protocol_df,
    note="Appendix protocol audit exported under the manuscript contract."
)

preproc_source_candidates = [TABLES_DIR / "table_preprocessing_and_tuning_audit.csv", TABLES_DIR / "table_paper_appendix_preprocessing_and_tuning_audit.csv"]
preproc_source = resolve_first_existing(preproc_source_candidates)
preproc_df = pd.read_csv(preproc_source) if preproc_source is not None else build_tex_preproc_tuning_table()
export_direct_table_asset(
    "tab:appendix_preproc_tuning_audit",
    PAPER_APPENDIX_DIR / "table_paper_appendix_preprocessing_and_tuning_audit.csv",
    source_candidates=preproc_source_candidates,
    dataframe=preproc_df,
    note="Appendix preprocessing and tuning audit exported under the manuscript contract." if preproc_source is not None else "Fallback reconstructed from the inline TeX appendix table."
)

bootstrap_source_candidates = [TABLES_DIR / "table_appendix_bootstrap_uncertainty_compact.csv"]
bootstrap_source = resolve_first_existing(bootstrap_source_candidates)
bootstrap_df = pd.read_csv(bootstrap_source) if bootstrap_source is not None else build_tex_bootstrap_table()
export_direct_table_asset(
    "tab:appendix_bootstrap_uncertainty",
    PAPER_APPENDIX_DIR / "table_appendix_bootstrap_uncertainty_compact.csv",
    source_candidates=bootstrap_source_candidates,
    dataframe=bootstrap_df,
    note="Appendix bootstrap uncertainty table exported under the manuscript contract." if bootstrap_source is not None else "Fallback reconstructed from the inline TeX appendix table."
)

split_context_source_candidates = [
    TABLES_DIR / "paper_appendix" / "table_paper_appendix_split_context_audit.csv",
    TABLES_DIR / "table_paper_appendix_split_context_audit.csv",
    TABLES_DIR / "table_appendix_split_context_audit_compact.csv",
    TABLES_DIR / "table_split_context_appendix.csv",
]
split_context_source = resolve_first_existing(split_context_source_candidates)
split_context_df = pd.read_csv(split_context_source) if split_context_source is not None else build_tex_split_context_table()
export_direct_table_asset(
    "tab:appendix_split_context_audit",
    PAPER_APPENDIX_DIR / "table_paper_appendix_split_context_audit.csv",
    source_candidates=split_context_source_candidates,
    dataframe=split_context_df,
    note="Appendix split/context audit exported under the manuscript contract." if split_context_source is not None else "Fallback reconstructed from the inline TeX appendix table."
)

cox_summary_source_candidates = [
    TABLES_DIR / "paper_appendix" / "table_paper_appendix_cox_ph_global_summary.csv",
    TABLES_DIR / "table_paper_appendix_cox_ph_global_summary.csv",
    TABLES_DIR / "table_appendix_cox_ph_global_summary.csv",
    TABLES_DIR / "table_cox_ph_global_summary.csv",
]
cox_summary_source = resolve_first_existing(cox_summary_source_candidates)
cox_summary_df = pd.read_csv(cox_summary_source) if cox_summary_source is not None else build_tex_cox_ph_summary_table()
export_direct_table_asset(
    "tab:appendix_cox_ph_summary",
    PAPER_APPENDIX_DIR / "table_paper_appendix_cox_ph_global_summary.csv",
    source_candidates=cox_summary_source_candidates,
    dataframe=cox_summary_df,
    note="Appendix Cox PH global summary exported under the manuscript contract." if cox_summary_source is not None else "Fallback reconstructed from the inline TeX appendix table."
)

cox_audit_source_candidates = [
    TABLES_DIR / "paper_appendix" / "table_paper_appendix_cox_ph_audit.csv",
    TABLES_DIR / "table_paper_appendix_cox_ph_audit.csv",
    TABLES_DIR / "table_appendix_cox_ph_audit.csv",
    TABLES_DIR / "table_cox_ph_assumption_audit.csv",
]
cox_audit_source = resolve_first_existing(cox_audit_source_candidates)
cox_ph_audit_df = pd.read_csv(cox_audit_source) if cox_audit_source is not None else build_tex_cox_ph_audit_table()
export_supporting_table_asset(
    "fig:appendix_cox_ph_diagnostics",
    PAPER_APPENDIX_DIR / "table_paper_appendix_cox_ph_audit.csv",
    source_candidates=cox_audit_source_candidates,
    dataframe=cox_ph_audit_df,
    note="Supporting PH audit table exported under the manuscript contract." if cox_audit_source is not None else "Fallback reconstructed from the manuscript narrative listing the flagged covariates."
)


### Funcao: generate_appendix_ph_figure

Definicao isolada de generate_appendix_ph_figure para reutilizacao nas etapas seguintes.


In [ ]:


def generate_appendix_ph_figure(output_path):
    if not MATPLOTLIB_AVAILABLE or cox_ph_audit_df is None:
        return False
    figure, axis = plt.subplots(figsize=(10, 4.8), constrained_layout=True)
    axis.axis("off")
    axis.text(0.01, 0.94, "Comparable Cox PH diagnostic summary", fontsize=18, fontweight="bold", va="top")
    axis.text(
        0.01,
        0.84,
        "Covariates with the strongest evidence of possible non-proportionality",
        fontsize=12,
        color="#264653",
        va="top",
    )
    display_df = cox_ph_audit_df.copy()
    if "Narrative rank" in display_df.columns and "Covariate" in display_df.columns:
        display_df = display_df.sort_values("Narrative rank")
        lines = [f"{int(row['Narrative rank'])}. {row['Covariate']}" for _, row in display_df.iterrows()]
        footer = "Narrative fallback generated from the manuscript because the quantitative PH audit table is not currently materialized in outputs_benchmark_survival/tables."
    else:
        covariate_col = pick_column(display_df, ["Covariate", "covariate", "feature", "variable"])
        lines = [f"- {value}" for value in display_df[covariate_col].astype(str).head(5)]
        footer = "Generated from the notebook PH audit table."
    y_position = 0.72
    for line in lines:
        axis.text(0.03, y_position, line, fontsize=12, va="top")
        y_position -= 0.10
    axis.text(
        0.01,
        0.10,
        "Global interpretation: localized departures rather than broad failure of proportional hazards.",
        fontsize=11,
        color="#6b705c",
        va="top",
    )
    axis.text(0.01, 0.03, footer, fontsize=9.5, color="#666666", va="bottom")
    figure.savefig(output_path, dpi=200, bbox_inches="tight")
    plt.close(figure)
    return True


In [ ]:


export_direct_figure_asset(
    "fig:appendix_cox_ph_diagnostics",
    PAPER_APPENDIX_DIR / tex_contract["fig:appendix_cox_ph_diagnostics"]["file_name"],
    source_candidates=[
        FIGURES_DIR / tex_contract["fig:appendix_cox_ph_diagnostics"]["file_name"],
        FIGURES_DIR / "paper_appendix" / tex_contract["fig:appendix_cox_ph_diagnostics"]["file_name"],
    ],
    generator=generate_appendix_ph_figure,
    note="Generated under the manuscript contract; falls back to a narrative diagnostic summary when the quantitative PH figure is unavailable."
)

for asset_row in tex_asset_rows:
    manifest_rows.append(
        {
            "tex_label": asset_row["tex_label"],
            "scope": asset_row["scope"],
            "artifact_type": asset_row["artifact_type"],
            "status": asset_row["status"],
            "file_name": asset_row["file_name"],
            "output_path": asset_row["output_path"],
        }
    )
for supporting_row in supporting_asset_rows:
    manifest_rows.append(
        {
            "tex_label": supporting_row["related_tex_label"],
            "scope": supporting_row["scope"],
            "artifact_type": supporting_row["artifact_type"],
            "status": supporting_row["status"],
            "file_name": supporting_row["file_name"],
            "output_path": supporting_row["output_path"],
        }
    )

tex_asset_registry_df = pd.DataFrame(tex_asset_rows).sort_values(["scope", "artifact_type", "tex_label"]).reset_index(drop=True)
supporting_asset_registry_df = pd.DataFrame(supporting_asset_rows).sort_values(["scope", "related_tex_label"]).reset_index(drop=True)
asset_manifest_df = pd.DataFrame(manifest_rows).sort_values(["scope", "artifact_type", "tex_label"]).reset_index(drop=True)

tex_asset_registry_path = METADATA_DIR / "paper_tex_asset_registry.csv"
supporting_registry_path = METADATA_DIR / "paper_tex_supporting_asset_registry.csv"
asset_manifest_path = METADATA_DIR / "paper_curated_asset_manifest.csv"
freeze_summary_path = METADATA_DIR / "paper_freeze_summary.json"

tex_asset_registry_df.to_csv(tex_asset_registry_path, index=False)
supporting_asset_registry_df.to_csv(supporting_registry_path, index=False)
asset_manifest_df.to_csv(asset_manifest_path, index=False)

freeze_summary = {
    "section_id": "G7",
    "contract_source": str(TEX_PATH),
    "paper_main_dir": str(PAPER_MAIN_DIR),
    "paper_appendix_dir": str(PAPER_APPENDIX_DIR),
    "n_expected_tex_assets": int(len(tex_asset_registry_df)),
    "n_exported_tex_assets": int((tex_asset_registry_df["status"] == "exported").sum()),
    "n_missing_tex_assets": int((tex_asset_registry_df["status"] == "missing").sum()),
    "n_exported_supporting_assets": int((supporting_asset_registry_df["status"] == "exported").sum()),
}
with open(freeze_summary_path, "w", encoding="utf-8") as file_handle:
    json.dump(freeze_summary, file_handle, indent=2)

print("Curated TeX output directories:")
print("-", PAPER_MAIN_DIR.resolve())
print("-", PAPER_APPENDIX_DIR.resolve())
print()
print("Direct TeX-linked assets:")
display(tex_asset_registry_df)
print("\nSupporting assets for TeX figures/tables:")
display(supporting_asset_registry_df)
missing_tex_assets_df = tex_asset_registry_df.loc[tex_asset_registry_df["status"] == "missing", ["tex_label", "file_name", "notes"]]
if not missing_tex_assets_df.empty:
    print("\nAssets still missing from the dedicated TeX export layer:")
    display(missing_tex_assets_df)
else:
    print("\nAll TeX-linked assets were materialized.")
print("\nSaved metadata files:")
print("-", tex_asset_registry_path.resolve())
print("-", supporting_registry_path.resolve())
print("-", asset_manifest_path.resolve())
print("-", freeze_summary_path.resolve())


# G8 — Display Curated Paper Figures

In [ ]:
# ==============================================================
# G8 — Display Curated Paper Figures
# --------------------------------------------------------------
# Purpose:
#   Load and display the curated TeX-facing figures exported in G7
#   directly inside the notebook for visual inspection.
#
# Methodological note:
#   This step is display-only.
#   No model is trained and no metric is recomputed.
# ==============================================================

print("\n" + "=" * 70)
print("G8 — Display Curated Paper Figures")
print("=" * 70)
print("Methodological note: this step displays curated TeX-facing figures only.")

required_names = ["OUTPUT_DIR", "METADATA_DIR"]
missing_names = [name for name in required_names if name not in globals()]
if missing_names:
    raise NameError(
        "Missing required objects from previous cells: "
        + ", ".join(missing_names)
        + ". Please run prior setup cells first."
    )

from IPython.display import Image, Markdown, display
import pandas as pd

registry_path = METADATA_DIR / "paper_tex_asset_registry.csv"
if not registry_path.exists():
    raise FileNotFoundError(
        f"TeX asset registry not found: {registry_path}. Please run Cell 136 first."
    )

registry_df = pd.read_csv(registry_path)
figure_registry_df = registry_df[
    (registry_df["artifact_type"] == "figure") & (registry_df["status"] == "exported")
].copy()

if figure_registry_df.empty:
    print("No exported TeX-facing figures are currently available.")
else:
    for _, row in figure_registry_df.sort_values(["scope", "tex_label"]).iterrows():
        display(Markdown(f"## {row['tex_label']}"))
        display(Markdown(row["tex_caption"]))
        display(Image(filename=row["output_path"]))
        print("-", Path(row["output_path"]).resolve())

missing_figures_df = registry_df[
    (registry_df["artifact_type"] == "figure") & (registry_df["status"] != "exported")
].copy()
if not missing_figures_df.empty:
    print("\nFigures still missing from the TeX export layer:")
    display(missing_figures_df[["tex_label", "file_name", "notes"]])

# G9 — Preview Curated Paper Evidence

This section previews the manuscript-facing evidence directly from the curated `paper_main` and `paper_appendix` directories created in G7.

In [ ]:
# ==============================================================
# G9 — Preview Curated Paper Evidence
# --------------------------------------------------------------
# Purpose:
#   Preview the manuscript-facing evidence directly from the curated
#   paper_main and paper_appendix directories created in G7.
#
# Methodological note:
#   This step is synthesis-only.
#   No model is trained and no metric is recomputed.
# ==============================================================

print("\n" + "=" * 70)
print("G9 — Preview Curated Paper Evidence")
print("=" * 70)
print("Methodological note: this step reads curated paper artifacts only.")

required_names = ["OUTPUT_DIR", "METADATA_DIR"]
missing_names = [name for name in required_names if name not in globals()]
if missing_names:
    raise NameError(
        "Missing required objects from previous cells: "
        + ", ".join(missing_names)
        + ". Please run prior setup cells first."
    )

import pandas as pd
from IPython.display import display

PAPER_MAIN_DIR = OUTPUT_DIR / "paper_main"
PAPER_APPENDIX_DIR = OUTPUT_DIR / "paper_appendix"
tex_registry_path = METADATA_DIR / "paper_tex_asset_registry.csv"
supporting_registry_path = METADATA_DIR / "paper_tex_supporting_asset_registry.csv"

if not tex_registry_path.exists():
    raise FileNotFoundError(
        f"TeX asset registry not found: {tex_registry_path}. Please run Cell 136 first."
    )

tex_registry_df = pd.read_csv(tex_registry_path)
supporting_registry_df = pd.read_csv(supporting_registry_path) if supporting_registry_path.exists() else pd.DataFrame()

print("\nDirect TeX-linked assets:")
display(tex_registry_df)

if not supporting_registry_df.empty:
    print("\nSupporting assets for TeX figures/tables:")
    display(supporting_registry_df)

exported_main_tables_df = tex_registry_df[
    (tex_registry_df["scope"] == "main")
    & (tex_registry_df["artifact_type"] == "table")
    & (tex_registry_df["status"] == "exported")
].copy()

exported_appendix_tables_df = tex_registry_df[
    (tex_registry_df["scope"] == "appendix")
    & (tex_registry_df["artifact_type"] == "table")
    & (tex_registry_df["status"] == "exported")
].copy()

print("\nPreview of exported main-paper tables:")
if exported_main_tables_df.empty:
    print("- No main-paper tables are currently materialized.")
else:
    for _, row in exported_main_tables_df.sort_values("tex_label").iterrows():
        preview_df = pd.read_csv(row["output_path"])
        print(f"- {row['tex_label']} -> {Path(row['output_path']).name} ({preview_df.shape[0]} rows)")
        display(preview_df)

print("\nPreview of exported appendix tables:")
if exported_appendix_tables_df.empty:
    print("- No appendix tables are currently materialized.")
else:
    for _, row in exported_appendix_tables_df.sort_values("tex_label").iterrows():
        preview_df = pd.read_csv(row["output_path"])
        print(f"- {row['tex_label']} -> {Path(row['output_path']).name} ({preview_df.shape[0]} rows)")
        display(preview_df)

missing_assets_df = tex_registry_df[tex_registry_df["status"] != "exported"].copy()
print("\nAssets still missing from the dedicated TeX export layer:")
if missing_assets_df.empty:
    print("- None. All TeX-linked assets are materialized.")
else:
    display(missing_assets_df[["tex_label", "artifact_type", "file_name", "notes"]])

print("\nExclusive TeX output directories:")
print("-", PAPER_MAIN_DIR.resolve())
print("-", PAPER_APPENDIX_DIR.resolve())